# ДЗ2 NLP: ensemble inference v11

Private, Internet off, одна T4/GPU0. Веса, decoder all, selector/vote, executor, RNG/OOM retry, sequence budget1024 и guards160/166/170/172 побайтно сохранены из v10. Единственное изменение политики: greedy C-long2x B6; K1=1/1/1 → K2=1/8/8, C/P1-B/W B6, T0.7/seed270042. После единогласного stage1 остановиться; иначе добавить7 P1-B и7 W. Максимум17 stochastic samples + greedy.

E1: frozen primary выбрана на tune1550/2048 до eval; eval1574 против baseline1559, Δ+0.732п.п. CI[+0.098;+1.367]. Прогноз137.322с — не cold timer. Новые формы/RNG могут изменить samples и качество. Диагностический expected_matches1553 оставлен из v10 для exact config delta; это прежний offline reference, не обещание v11-score.

Run All: ровно один cold test1919 с полным timer и отдельный свежий tune2048; timeout720, watchdog660/690. Скачайте из ensemble-inference-v11/: test-1/answers.jsonl, test-1/timing.json, test-1/summary.json, summary.json. Для проверки exact code parity сохраните исполненный notebook; для независимого replay tune дополнительно tune/answers.jsonl, tune/candidate-journal.json.gz, tune/summary.json, tune/quality.json. Ответ всегда test-1, не выбирается по tune. Обучения нет.

In [ ]:
import time,os,sys,json
from pathlib import Path
SESSION_START=time.monotonic()
os.environ['CUDA_VISIBLE_DEVICES']='0'
os.environ['TOKENIZERS_PARALLELISM']='false'
OUTPUT=Path('/kaggle/working/ensemble-inference-v11') if Path('/kaggle/input').exists() else Path('ensemble-inference-v11')
OUTPUT.mkdir(parents=True,exist_ok=True)
assert not (OUTPUT/'answers.jsonl').exists(),'Use a new OUTPUT directory for another session; saved answers are retained'
MODULES=OUTPUT/'src';MODULES.mkdir(exist_ok=True)
SOURCES={'final_core.py': "# Generated by build_final_inference.py from exp26_core.py; do not edit.\nimport math,time\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nPAD,BOS,EOS,TOKEN_OFFSET=0,1,2,3\nVOCAB=8192\nMAX_POSITIONS=2048\nSEED=42\n\nclass SwiGLUInput(nn.Module):\n def __init__(self,width,hidden):\n  super().__init__();self.gate=nn.Linear(width,hidden);self.value=nn.Linear(width,hidden)\n def forward(self,x):return F.silu(self.gate(x))*self.value(x)\n\n\ndef replace_ffn_with_swiglu(transformer,width,hidden):\n with torch.random.fork_rng(devices=[]):\n  torch.set_rng_state(torch.Generator().manual_seed(SEED).get_state())\n  for layer in list(transformer.encoder.layers)+list(transformer.decoder.layers):\n   layer.linear1=SwiGLUInput(width,hidden);layer.linear2=nn.Linear(hidden,width);layer.activation=nn.Identity()\n   if isinstance(layer,nn.TransformerEncoderLayer):layer.activation_relu_or_gelu=0\n   for module in [layer.linear1.gate,layer.linear1.value,layer.linear2]:nn.init.xavier_uniform_(module.weight)\n\n\nclass SimpleTransformer(nn.Module):\n def __init__(self,d_model=256,heads=4,encoder_layers=4,decoder_layers=4,swiglu_dim=680,\n              vocab=VOCAB,max_positions=MAX_POSITIONS,dropout=.1,qk_norm=False):\n  super().__init__();self.width=d_model;self.vocab=vocab\n  self.tokens=nn.Embedding(vocab,d_model,padding_idx=PAD);self.positions=nn.Embedding(max_positions,d_model)\n  self.transformer=nn.Transformer(d_model=d_model,nhead=heads,num_encoder_layers=encoder_layers,\n   num_decoder_layers=decoder_layers,dim_feedforward=4*d_model,dropout=dropout,batch_first=True,norm_first=True)\n  self.transformer.encoder.enable_nested_tensor=False;self.transformer.encoder.use_nested_tensor=False\n  self.head=nn.Linear(d_model,vocab);replace_ffn_with_swiglu(self.transformer,d_model,swiglu_dim)\n  if qk_norm:\n   from qk_attention import install_qk_norm\n   install_qk_norm(self.transformer,qk_norm)\n def embed(self,ids):\n  if ids.shape[1]>self.positions.num_embeddings:raise ValueError('Position overflow')\n  return self.tokens(ids)*math.sqrt(self.width)+self.positions(torch.arange(ids.shape[1],device=ids.device))\n def decode_prefix(self,tgt,memory,source_padding):\n  mask=torch.ones(tgt.shape[1],tgt.shape[1],device=tgt.device,dtype=torch.bool).triu(1)\n  return self.transformer.decoder(self.embed(tgt),memory,tgt_mask=mask,tgt_key_padding_mask=tgt.eq(PAD),memory_key_padding_mask=source_padding)\n def forward(self,src,tgt):\n  memory=self.transformer.encoder(self.embed(src),src_key_padding_mask=src.eq(PAD))\n  return self.head(self.decode_prefix(tgt,memory,src.eq(PAD)))\n @torch.inference_mode()\n def generate(self,src,max_new_tokens=512):\n  self.eval();memory=self.transformer.encoder(self.embed(src),src_key_padding_mask=src.eq(PAD))\n  tgt=torch.full((len(src),1),BOS,device=src.device,dtype=torch.long);done=torch.zeros(len(src),device=src.device,dtype=torch.bool)\n  for _ in range(max_new_tokens):\n   logits=self.head(self.decode_prefix(tgt,memory,src.eq(PAD))[:,-1]);logits[:,[PAD,BOS]]=-torch.inf\n   token=logits.argmax(-1).masked_fill(done,PAD);done|=token.eq(EOS);tgt=torch.cat([tgt,token[:,None]],dim=1)\n   if bool(done.all()):break\n  return tgt[:,1:]\n\n\ndef encode(tokenizer,text):return [i+TOKEN_OFFSET for i in tokenizer.encode(text,add_special_tokens=False).ids]\n\n\ndef decode_bytes(tokenbytes,ids):\n chunks=[]\n for i in ids:\n  if i==EOS:break\n  if i>=TOKEN_OFFSET:chunks.append(tokenbytes[i])\n return b''.join(chunks)\n\n\ndef active_tokens(ids):\n ids=[int(i)for i in ids if i!=PAD]\n return ids[:ids.index(EOS)+1]if EOS in ids else ids\n\n\nclass BudgetExpired(Exception):pass\n\n\ndef check_deadline(deadline):\n if deadline is not None and time.monotonic()>=deadline:raise BudgetExpired('Session budget reached; completed run boundaries are preserved')\n\n\ndef sync(device):\n if device.type=='cuda':torch.cuda.synchronize(device)\n\n", 'final_fast.py': '# Generated by build_final_inference.py from exp27b_fast.py; do not edit.\nimport math\nimport torch\nfrom torch.nn import functional as F\nimport final_core as C\n\nclass Context:\n    @torch.inference_mode()\n    def __init__(self,model,src,capacity):\n        self.model,self.src,self.capacity=model,src,capacity\n        self.width=model.width;self.heads=model.transformer.decoder.layers[0].self_attn.num_heads;self.dim=self.width//self.heads\n        memory=model.transformer.encoder(model.embed(src),src_key_padding_mask=src.eq(C.PAD))\n        self.allowed=src.ne(C.PAD)[:,None,None,:];self.positions=model.positions(torch.arange(capacity,device=src.device));self.cross=[]\n        for layer in model.transformer.decoder.layers:\n            a=layer.multihead_attn;k,v=F.linear(memory,a.in_proj_weight[self.width:],a.in_proj_bias[self.width:]).chunk(2,-1)\n            kh=k.reshape(len(src),-1,self.heads,self.dim).transpose(1,2).contiguous()\n            if hasattr(a,"k_norm"):kh=a.k_norm(kh)\n            self.cross.append((kh,v.reshape(len(src),-1,self.heads,self.dim).transpose(1,2).contiguous()))\n\n\nclass Decoder:\n    @torch.inference_mode()\n    def __init__(self,context,k):\n        self.c,self.k,self.t=context,k,0;self.n=len(context.src)*k\n        self.allowed=context.allowed.repeat_interleave(k,dim=0)\n        self.cross=[tuple(t.repeat_interleave(k,dim=0).contiguous() for t in pair) for pair in context.cross]\n        self.cross_bytes=sum(t.numel()*t.element_size() for pair in self.cross for t in pair)\n        shape=(self.n,context.heads,context.capacity,context.dim)\n        self.self_kv=[(context.positions.new_empty(shape),context.positions.new_empty(shape)) for _ in context.cross]\n        self.valid=torch.zeros(self.n,context.capacity,dtype=torch.bool,device=context.src.device)\n    def heads(self,x):return x.reshape(self.n,1,self.c.heads,self.c.dim).transpose(1,2)\n    def attend(self,a,q,k,v,mask):\n        assert q.ndim==k.ndim==v.ndim==4\n        if hasattr(a,"q_norm"):q=a.q_norm(q)\n        y=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,dropout_p=0.,is_causal=False)\n        return a.out_proj(y.transpose(1,2).contiguous().reshape(self.n,1,self.c.width))\n    @torch.inference_mode()\n    def step(self,tokens):\n        c=self.c;t=self.t;x=c.model.tokens(tokens[:,None])*math.sqrt(c.width)+c.positions[t][None,None,:]\n        self.valid[:,t]=tokens.ne(C.PAD)\n        for layer,(sk,sv),(ck,cv) in zip(c.model.transformer.decoder.layers,self.self_kv,self.cross):\n            a=layer.self_attn;q,k,v=F.linear(layer.norm1(x),a.in_proj_weight,a.in_proj_bias).chunk(3,-1)\n            sk[:,:,t:t+1]=a.k_norm(self.heads(k)) if hasattr(a,"k_norm") else self.heads(k);sv[:,:,t:t+1]=self.heads(v)\n            x=x+layer.dropout1(self.attend(a,self.heads(q),sk[:,:,:t+1],sv[:,:,:t+1],self.valid[:,None,None,:t+1]))\n            a=layer.multihead_attn;q=F.linear(layer.norm2(x),a.in_proj_weight[:c.width],a.in_proj_bias[:c.width])\n            x=x+layer.dropout2(self.attend(a,self.heads(q),ck,cv,self.allowed))\n            x=x+layer.dropout3(layer.linear2(layer.dropout(layer.activation(layer.linear1(layer.norm3(x))))))\n        if c.model.transformer.decoder.norm is not None:x=c.model.transformer.decoder.norm(x)\n        self.t+=1;return c.model.head(x[:,0])\n\n\nfrom final_decode import decode\n', 'final_choose.py': "# Generated by build_final_inference.py from exp27_decode.py; do not edit.\nimport math\n\ndef choose(row,cfg):\n    greedy=row['greedy']\n    if cfg['mode']=='greedy': return greedy\n    samples=row['samples'][str(cfg['temperature'])][:cfg['k']]\n    if cfg['mode']=='fallback':\n        return greedy if greedy['execution']['status']=='numeric' else next((s for s in samples if s['execution']['status']=='numeric'),greedy)\n    votes=([greedy] if cfg['greedy_vote'] else [])+samples\n    good=[v for v in votes if v['execution']['status']=='numeric']\n    if not good:return greedy\n    # isclose is not transitive: explicitly use connected components of its pairwise graph.\n    parent=list(range(len(good)))\n    def root(i):\n        while parent[i]!=i:parent[i]=parent[parent[i]];i=parent[i]\n        return i\n    for i in range(len(good)):\n        for j in range(i):\n            if math.isclose(good[i]['execution']['value'],good[j]['execution']['value'],rel_tol=1e-6,abs_tol=1e-6):\n                parent[root(i)]=root(j)\n    clusters={}\n    for i,v in enumerate(good):clusters.setdefault(root(i),[]).append(v)\n    def key(vs):\n        contains=greedy['execution']['status']=='numeric' and any(math.isclose(v['execution']['value'],greedy['execution']['value'],rel_tol=1e-6,abs_tol=1e-6) for v in vs)\n        return len(vs),contains,sum(v['mean_logprob'] for v in vs)/len(vs)\n    winning=max(clusters.values(),key=key)\n    return max(winning,key=lambda v:v['mean_logprob'])\n\n", 'final_split.py': "# Generated by build_final_inference.py from l2_sampler.py; do not edit.\nimport hashlib\n\ndef fixed_split(n=4096):\n    ordered = sorted(range(n), key=lambda i: hashlib.sha256(\n        f'candidate-selection-v1:heldout4k:{i}'.encode()).digest())\n    return {'tune2048': sorted(ordered[:n//2]), 'eval2048': sorted(ordered[n//2:])}\n\n", 'final_representation.py': "# Generated by build_final_inference.py from s3_representation.py; do not edit.\nfrom tokenizers import Tokenizer,AddedToken\nMODES={'header_tail':'<|header_tail|>','header':'<|header|>','raw':'<|raw|>'}\nNUMBERS=[f'<|N{i}|>' for i in range(1,21)]\n\ndef restore_compact(text):\n    mode = next((k for k, token in MODES.items() if text.startswith(token)), None)\n    if mode is None:\n        raise ValueError('Missing representation mode')\n    fragment = text[len(MODES[mode]):]\n    if mode == 'raw':\n        return fragment\n    if mode == 'header':\n        return 'def solve() -> int:\\n' + fragment\n    lines = fragment.splitlines(keepends=True)\n    if not lines or not lines[-1].startswith('    '):\n        raise ValueError('Missing final result expression')\n    lines[-1] = '    result = ' + lines[-1][4:]\n    body = ''.join(lines)\n    return 'def solve() -> int:\\n' + body + ('' if body.endswith('\\n') else '\\n') + '    return result\\n'\n\n\ndef tokenizer(path, variant):\n    tok = Tokenizer.from_file(str(path))\n    assert tok.get_vocab_size() == 8189\n    added = list(MODES.values()) if variant == 30 else NUMBERS\n    assert tok.add_special_tokens([AddedToken(t, normalized=False, special=True) for t in added]) == len(added)\n    # Original byte alphabet is unchanged; added specials decode literally, never get skipped.\n    visible = list(range(33,127)) + list(range(161,173)) + list(range(174,256))\n    bchars = {b:chr(b) for b in visible}\n    for i,b in enumerate(b for b in range(256) if b not in bchars):\n        bchars[b] = chr(256+i)\n    charbytes = {c:b for b,c in bchars.items()}\n    tokenbytes = [b'',b'',b''] + [bytes(charbytes[c] for c in tok.id_to_token(i)) for i in range(8189)]\n    tokenbytes += [t.encode() for t in added]\n    assert all(tok.encode(t, add_special_tokens=False).ids == [8189+i] for i,t in enumerate(added))\n    return tok, tokenbytes\n\n", 'final_decode.py': '"""V3 decode loop; frozen Context/Decoder mathematics live in final_fast."""\nimport torch\nfrom torch.nn import functional as F\nimport final_core as C\n\n\n@torch.inference_mode()\ndef decode(context, k=1, temperature=None, deadline=None, check_every=16, compute_logprob=True):\n    from final_fast import Decoder\n    d = Decoder(context, k)\n    device = context.src.device\n    token = torch.full((d.n,), C.BOS, device=device, dtype=torch.long)\n    done = torch.zeros(d.n, dtype=torch.bool, device=device)\n    # KV, output, running LP/length buffers are allocated once, never concatenated.\n    output = torch.full((d.n, context.capacity), C.PAD, device=device, dtype=torch.long)\n    sums = torch.zeros(d.n, device=device)\n    lengths = torch.zeros(d.n, dtype=torch.long, device=device)\n    for t in range(context.capacity):\n        if t % check_every == 0:\n            C.check_deadline(deadline)\n        logits = d.step(token)\n        if temperature is not None or compute_logprob:\n            logits = logits.float()\n        logits[:, :2] = -torch.inf  # PAD/BOS: contiguous slice avoids a per-step index tensor.\n        pick = logits.argmax(-1) if temperature is None else torch.multinomial(\n            F.softmax(logits / temperature, -1), 1).squeeze(-1)\n        if compute_logprob:\n            logprob = F.log_softmax(logits, -1).gather(-1, pick[:, None]).squeeze(-1)\n            sums.add_(torch.where(done, 0., logprob))\n        lengths.add_(~done)\n        token = pick.masked_fill(done, C.PAD)\n        output[:, t] = token\n        done |= token.eq(C.EOS)\n        if (t + 1) % check_every == 0 and bool(done.all().cpu()):\n            break\n    return (output[:, :d.t].reshape(len(context.src), k, d.t).cpu(),\n            (sums / lengths.clamp_min(1)).reshape(-1, k).cpu(), d.cross_bytes)\n', 'final_numeric.py': '"""Final-only numeric policy; historical experiment executors remain unchanged."""\nimport ast\nimport builtins\nimport math\nimport multiprocessing as mp\nimport os\nimport random\nimport resource\nimport signal\nimport sys\nimport time\nfrom collections import Counter, deque\nfrom types import SimpleNamespace\n\nSAFE_BUILTINS = {n: getattr(builtins, n) for n in (\n    \'abs\',\'all\',\'any\',\'bool\',\'dict\',\'divmod\',\'enumerate\',\'float\',\'int\',\'len\',\'list\',\n    \'max\',\'min\',\'pow\',\'range\',\'reversed\',\'round\',\'set\',\'sorted\',\'str\',\'sum\',\'tuple\',\'zip\',\n    \'Exception\',\'ValueError\',\'TypeError\',\'ZeroDivisionError\',\'OverflowError\',\'ArithmeticError\')}\nFORBIDDEN_NAMES = {\'open\',\'input\',\'eval\',\'exec\',\'compile\',\'globals\',\'locals\',\'vars\',\'getattr\',\n    \'setattr\',\'delattr\',\'dir\',\'help\',\'breakpoint\',\'exit\',\'quit\',\'os\',\'sys\',\'subprocess\',\n    \'socket\',\'ctypes\',\'importlib\',\'builtins\',\'resource\',\'signal\',\'pickle\',\'marshal\',\n    \'memoryview\',\'type\',\'object\',\'super\'}\nFORBIDDEN_ATTRS = {\'mro\',\'subclasses\',\'format_map\',\'gi_frame\',\'f_globals\',\'f_locals\',\n    \'tb_frame\',\'func_globals\',\'code\',\'modules\',\'system\',\'popen\',\'load\',\'loads\',\'dump\',\'dumps\',\'random\',\'format\'}\nSAFE_BUILTINS[\'print\']=lambda *args,**kwargs:None  # no program output or file handle\nALLOWED_IMPORTS = {\'math\',\'sympy\'}\nSYM_NAMES = {\'Symbol\',\'symbols\',\'Rational\',\'Integer\',\'Float\',\'Eq\',\'solve\',\'sqrt\',\'simplify\'}\nMAX_CODE_BYTES=100000\nMAX_AST_NODES=5000\n\nclass ProgramTimeout(Exception): pass\n\ndef filter_program(code):\n    if not isinstance(code,str) or len(code.encode(\'utf-8\'))>MAX_CODE_BYTES:\n        return None,\'blocked_size\'\n    try: tree=ast.parse(code)\n    except (SyntaxError,ValueError,RecursionError): return None,\'syntax_error\'\n    nodes=list(ast.walk(tree))\n    if len(nodes)>MAX_AST_NODES:return None,\'blocked_size\'\n    for n in nodes:\n        if isinstance(n,ast.Attribute) and isinstance(n.ctx,(ast.Store,ast.Del)):\n            return None,\'blocked_attribute_mutation\'\n        if isinstance(n,ast.Name) and (n.id.startswith(\'__\') or n.id in FORBIDDEN_NAMES):\n            return None,\'blocked_name\'\n        if isinstance(n,ast.Attribute) and (n.attr.startswith(\'_\') or n.attr in FORBIDDEN_ATTRS):\n            return None,\'blocked_attribute\'\n        if isinstance(n,(ast.ClassDef,ast.AsyncFunctionDef,ast.Await,ast.Yield,ast.YieldFrom)):\n            return None,\'blocked_construct\'\n        if isinstance(n,(ast.FunctionDef,ast.arg)):\n            name=n.name if isinstance(n,ast.FunctionDef) else n.arg\n            if name.startswith(\'__\') or name in FORBIDDEN_NAMES:return None,\'blocked_name\'\n        if isinstance(n,ast.Import):\n            if any(a.name not in ALLOWED_IMPORTS or (a.asname and a.asname.startswith(\'_\'))for a in n.names):\n                return None,\'blocked_import\'\n        if isinstance(n,ast.ImportFrom):\n            if n.level or n.module not in ALLOWED_IMPORTS:return None,\'blocked_import\'\n            if any(a.name.startswith(\'_\') or a.name==\'*\'or (a.asname and a.asname.startswith(\'_\'))for a in n.names):\n                return None,\'blocked_import\'\n            if n.module==\'sympy\' and any(a.name not in SYM_NAMES for a in n.names):return None,\'blocked_import\'\n    return tree,None\n\ndef safe_sympy():\n    import sympy\n    # All sympify bindings import this parser dynamically, including object methods.\n    # Symbol names use the separately validated symbols() syntax, never parse_expr.\n    from sympy.parsing import sympy_parser\n    def denied_parser(*args,**kwargs):\n        raise ValueError(\'String expression parsing forbidden\')\n    sympy_parser.parse_expr=denied_parser\n    from sympy.core.cache import clear_cache\n    clear_cache()\n    from sympy.core.random import seed\n    seed(0)\n    def no_strings(x):\n        if isinstance(x,str):raise ValueError(\'String expressions forbidden\')\n        if isinstance(x,(tuple,list)):\n            for y in x:no_strings(y)\n        if isinstance(x,dict):\n            for k,v in x.items():no_strings(k);no_strings(v)\n    def wrapped(function):\n        def call(*args,**kwargs):\n            no_strings(args);no_strings(list(kwargs.values()))\n            return function(*args,**kwargs)\n        return call\n    def symbol(name,*args,**kwargs):\n        if not isinstance(name,str) or not name or not name.replace(\'_\',\'\').isalnum():raise ValueError(\'Bad symbol\')\n        return sympy.Symbol(name,*args,**kwargs)\n    def symbols(names,*args,**kwargs):\n        if not isinstance(names,str)or any(not(c.isalnum()or c in \' _,:\')for c in names):raise ValueError(\'Bad symbols\')\n        return sympy.symbols(names,*args,**kwargs)\n    return SimpleNamespace(Symbol=symbol,symbols=symbols,\n        **{n:wrapped(getattr(sympy,n))for n in SYM_NAMES-{\'Symbol\',\'symbols\'}})\n\n_MODULES={}\ndef safe_import(name,globals=None,locals=None,fromlist=(),level=0):\n    if level or name not in ALLOWED_IMPORTS:raise ImportError(\'Blocked import\')\n    if name not in _MODULES:\n        _MODULES[name]=SimpleNamespace(**{n:getattr(math,n)for n in dir(math)if not n.startswith(\'_\')})if name==\'math\'else safe_sympy()\n    return _MODULES[name]\n\ndef worker_init(memory_bytes):\n    # Spawned workers import no Torch. No GPU state or model is inherited.\n    resource.setrlimit(resource.RLIMIT_CORE,(0,0))\n    if sys.platform.startswith(\'linux\') and hasattr(resource,\'RLIMIT_AS\'):\n        soft,hard=resource.getrlimit(resource.RLIMIT_AS)\n        value=memory_bytes if hard==resource.RLIM_INFINITY else min(memory_bytes,hard)\n        resource.setrlimit(resource.RLIMIT_AS,(value,value))\n    signal.signal(signal.SIGALRM,lambda signum,frame:(_ for _ in ()).throw(ProgramTimeout()))\n\ndef prepare_pristine_worker():\n    """Run once in a Torch-free, single-threaded parent; never run user code here."""\n    _MODULES.clear()\n    safe_import(\'math\')\n    safe_import(\'sympy\')  # Parser deny patch, cache clear and SymPy seed happen ONCE.\n    random.seed(0)\n\ndef execute_task(payload, tree=None, isolated=False):\n    code,wall_limit,cpu_limit=payload\n    started=time.monotonic()\n    if not isolated:\n        _MODULES.clear()  # Historical/reference path; fork children inherit pristine modules.\n    random.seed(0)\n    if tree is None:\n        tree,status=filter_program(code)\n        if status:return {\'status\':status}\n    # RLIMIT_CPU is cumulative per process. Set a new SOFT deadline for EACH task;\n    # retain the hard limit so the soft deadline can increase for reused workers.\n    used=resource.getrusage(resource.RUSAGE_SELF)\n    soft=max(1,math.ceil(used.ru_utime+used.ru_stime)+int(math.ceil(cpu_limit)))\n    _,hard=resource.getrlimit(resource.RLIMIT_CPU)\n    if hard!=resource.RLIM_INFINITY:soft=min(soft,hard)\n    resource.setrlimit(resource.RLIMIT_CPU,(soft,hard))\n    signal.signal(signal.SIGPROF,lambda *_:(_ for _ in ()).throw(ProgramTimeout()))\n    signal.setitimer(signal.ITIMER_PROF,cpu_limit)\n    signal.setitimer(signal.ITIMER_REAL,wall_limit)\n    try:\n        env={\'__builtins__\':dict(SAFE_BUILTINS,__import__=safe_import)}\n        exec(compile(tree,\'<bounded-dataset-program>\',\'exec\'),env,env)\n        solve=env.get(\'solve\')\n        if not callable(solve):return {\'status\':\'missing_solve\'}\n        value=solve()\n        if isinstance(value,bool):value=int(value)\n        if not isinstance(value,(int,float)):\n            # SymPy real scalars may be used; containers/strings/complex refused.\n            if type(value).__module__.startswith(\'sympy\') and value.is_real is True:value=float(value)\n            else:return {\'status\':\'non_number\'}\n        value=float(value)\n        if not math.isfinite(value):return {\'status\':\'non_finite\'}\n        return {\'status\':\'numeric\',\'value\':value}\n    except ProgramTimeout:return {\'status\':\'timeout\'}\n    except BaseException as e:return {\'status\':\'execution_error\',\'error_type\':type(e).__name__}\n    finally:\n        signal.setitimer(signal.ITIMER_REAL,0)\n        signal.setitimer(signal.ITIMER_PROF,0)\n', 'final_vote.py': '"""Frozen mixed vote. final_choose is generated verbatim from exp27_decode.choose."""\nimport json\nimport math\nfrom pathlib import Path\nfrom final_choose import choose\n\nCONFIG = {\'mode\': \'vote\', \'temperature\': .7, \'k\': 8, \'greedy_vote\': True}\n\n\ndef select(greedy, main, long):\n    # Deduplication only caches execution: repeated samples still cast votes.\n    samples = [c for c in main[:4] + long[:4] if c.get(\'mean_logprob\') is not None\n               and math.isfinite(c[\'mean_logprob\'])]\n    row = {\'greedy\': greedy, \'samples\': {\'0.7\': samples}}\n    winner = choose(row, CONFIG)\n    if winner[\'execution\'][\'status\'] != \'numeric\':\n        return greedy\n    # Code representative: greedy anywhere in the winning connected component,\n    # otherwise greatest mean LP, stable order. Do not replace graph connectivity\n    # by a direct isclose comparison (isclose is non-transitive).\n    good = [c for c in [greedy] + samples if c[\'execution\'][\'status\'] == \'numeric\']\n    component = [winner]\n    remaining = [c for c in good if c is not winner]\n    while True:\n        adjacent = [c for c in remaining if any(math.isclose(\n            c[\'execution\'][\'value\'], v[\'execution\'][\'value\'], rel_tol=1e-6, abs_tol=1e-6)\n            for v in component)]\n        if not adjacent:\n            break\n        component.extend(adjacent)\n        adjacent_ids = {id(c) for c in adjacent}\n        remaining = [c for c in remaining if id(c) not in adjacent_ids]\n    if any(c is greedy for c in component):\n        return greedy\n    return winner\n\n\ndef select_available(record):\n    """A missing model/batch never discards already executed samples or their votes."""\n    available = {key: [c for c in record[key]\n                       if c[\'execution\'][\'status\'] != \'budget_not_evaluated\']\n                 for key in (\'main\', \'long\')}\n    count = len(available[\'main\']) + len(available[\'long\'])\n    return select(record[\'greedy\'], available[\'main\'], available[\'long\']), count\n\n\ndef write_answers(path, codes):\n    with Path(path).open(\'w\', encoding=\'utf8\') as stream:\n        for code in codes:\n            if not isinstance(code, str):\n                raise TypeError(\'Answer code must be a string\')\n            stream.write(json.dumps({\'code\': code}, ensure_ascii=False) + \'\\n\')\n\n\ndef validate_answers(path, expected):\n    with Path(path).open(encoding=\'utf8\') as stream:\n        rows = [json.loads(line) for line in stream]\n    assert len(rows) == expected\n    assert all(set(r) == {\'code\'} and isinstance(r[\'code\'], str) for r in rows)\n    return rows\n\n\ndef verify_winners(winners, deadline, workers=None):\n    """Fresh service: compare the actual chosen CODE against its voted execution."""\n    from final_executor import Executor\n    executor = Executor(workers)\n    eligible = [i for i, w in enumerate(winners) if w[\'execution\'][\'status\'] == \'numeric\']\n    items = [{\'code\': winners[i][\'code\']} for i in eligible]\n    try:\n        executor.submit(items)\n        complete = executor.drain(deadline)\n        executor.attach(items)\n        mismatches = [i for i, item in zip(eligible, items) if\n                      item[\'execution\'][\'status\'] != \'numeric\' or not math.isclose(\n                          item[\'execution\'][\'value\'], winners[i][\'execution\'][\'value\'],\n                          rel_tol=1e-6, abs_tol=1e-6)]\n        return {\'checked_rows\': len(eligible), \'complete\': complete,\n                \'mismatch_rows\': mismatches, \'execution\': executor.summary()}\n    finally:\n        executor.close()\n', 'final_execute.py': '"""Persistent Torch-free service: fresh numeric namespaces, pristine SymPy forks.\n\nstdin: JSON lists of [id, code], or null to finish. stdout: [id, execution, metrics].\nThe launcher can kill the complete service process group, including its workers.\n"""\nimport json\nimport multiprocessing as mp\nimport os\nimport queue\nimport resource\nimport select\nimport signal\nimport sys\nimport threading\nimport time\nfrom collections import deque\nfrom multiprocessing.connection import wait\nimport final_numeric as numeric\n\nMEMORY = 768 * 1024**2\nWALL = 3.\nCPU = 2.\n\n\ndef kill_program(pid):\n    # Each program owns a group; kill descendants even after a successful result.\n    try:\n        os.killpg(pid, signal.SIGKILL)\n    except (ProcessLookupError, PermissionError):\n        try:\n            os.kill(pid, signal.SIGKILL)  # Child may not yet have called setsid().\n        except ProcessLookupError:\n            pass\n    until = time.monotonic()+.1\n    while time.monotonic() < until:\n        try:\n            if os.waitpid(pid, os.WNOHANG)[0]:\n                return\n        except ChildProcessError:\n            return\n        time.sleep(.001)\n    raise RuntimeError(\'Program could not be reaped; replace pristine worker\')\n\n\ndef isolated_task(code, tree, connection):\n    reader, writer = os.pipe()\n    parent = os.getpid()\n    pid = os.fork()  # This parent has no Torch/CUDA and no reader/BLAS threads.\n    if pid == 0:\n        try:\n            os.close(reader)\n            connection.close()\n            os.setsid()\n            if sys.platform.startswith(\'linux\'):\n                import ctypes\n                signal.signal(signal.SIGTERM, lambda *_: os.killpg(os.getpgrp(), signal.SIGKILL))\n                if ctypes.CDLL(None).prctl(1, signal.SIGTERM, 0, 0, 0) != 0 or os.getppid() != parent:\n                    os.killpg(os.getpgrp(), signal.SIGKILL)\n            resource.setrlimit(resource.RLIMIT_CPU, (int(CPU)+1, int(CPU)+1))\n            result = numeric.execute_task((code, WALL, CPU), tree=tree, isolated=True)\n            os.write(writer, json.dumps(result, separators=(\',\', \':\')).encode())\n        finally:\n            os._exit(0)\n    os.close(writer)\n    try:\n        # Same outer limit as v2; catches code that repeatedly catches timer exceptions.\n        if select.select([reader], [], [], WALL+2.)[0]:\n            data = os.read(reader, 4096)\n            if data:\n                return json.loads(data)\n        return {\'status\': \'timeout_or_worker_exit\'}\n    finally:\n        os.close(reader)\n        kill_program(pid)\n\n\ndef worker(connection):\n    # Library diagnostics and user output cannot corrupt the JSON protocol.\n    with open(os.devnull, \'w\') as sink:\n        os.dup2(sink.fileno(), 1)\n        os.dup2(sink.fileno(), 2)\n    numeric.worker_init(MEMORY)\n    pristine = None\n    while True:\n        item = connection.recv()\n        if item is None:\n            return\n        index, code = item\n        started = time.monotonic()\n        used = time.process_time()\n        before = resource.getrusage(resource.RUSAGE_CHILDREN)\n        tree, status = numeric.filter_program(code)\n        if status:\n            result = {\'status\': status}\n        elif any((isinstance(node, numeric.ast.Import) and any(a.name == \'sympy\' for a in node.names)) or\n                 (isinstance(node, numeric.ast.ImportFrom) and node.module == \'sympy\')\n                 for node in numeric.ast.walk(tree)):\n            if pristine is None:\n                numeric.prepare_pristine_worker()\n                pristine = dict(numeric._MODULES)\n            numeric._MODULES.clear()\n            numeric._MODULES.update(pristine)\n            result = isolated_task(code, tree, connection)\n        else:\n            # Only builtins and a fresh math namespace are reachable on this path.\n            # No user code can access SymPy objects/caches; env and timers are per task.\n            result = numeric.execute_task((code, WALL, CPU), tree=tree)\n        after = resource.getrusage(resource.RUSAGE_CHILDREN)\n        connection.send((index, result, {\'task_wall_seconds\': time.monotonic()-started,\n            \'task_cpu_seconds\': time.process_time()-used + after.ru_utime+after.ru_stime-before.ru_utime-before.ru_stime}))\n\n\ndef serve(workers):\n    inbox = queue.Queue()\n    def read():\n        for line in sys.stdin:\n            inbox.put(json.loads(line))\n        inbox.put(None)\n    threading.Thread(target=read, daemon=True).start()\n    context = mp.get_context(\'spawn\')\n    slots = []\n    pending = deque()\n    finished = False\n    def start():\n        parent, child = context.Pipe()\n        process = context.Process(target=worker, args=(child,), daemon=True)\n        process.start()\n        child.close()\n        return [process, parent, None, 0.]\n    def stop(slot):\n        process, pipe, _, _ = slot\n        if process.is_alive():\n            process.kill()\n        process.join(timeout=.05)\n        pipe.close()\n    def emit(index, result, metrics=None):\n        print(json.dumps([index, result, metrics]), flush=True)\n    try:\n        slots = [start() for _ in range(workers)]\n        while not finished or pending or any(s[2] is not None for s in slots):\n            while True:\n                try:\n                    batch = inbox.get_nowait()\n                except queue.Empty:\n                    break\n                if batch is None:\n                    finished = True\n                else:\n                    pending.extend(batch)\n            for number, slot in enumerate(slots):\n                process, pipe, active, submitted = slot\n                pipe_failed = False\n                # Worker enforces wall+2; allow a bounded IPC/reap margin here.\n                try:\n                    if active is not None and pipe.poll():\n                        index, result, *metadata = pipe.recv()\n                        assert index == active\n                        emit(index, result, metadata[0] if metadata else None)\n                        slot[2] = None\n                except (EOFError, OSError):\n                    pipe_failed = True\n                if pipe_failed or not process.is_alive() or (slot[2] is not None and time.monotonic()-submitted >= WALL+2.5):\n                    if slot[2] is not None:\n                        emit(slot[2], {\'status\': \'timeout_or_worker_exit\'})\n                    stop(slot)\n                    slots[number] = slot = start()\n                    process, pipe = slot[:2]\n                if slot[2] is None and pending:\n                    index, code = pending.popleft()\n                    try:\n                        pipe.send((index, code))\n                        slot[2:] = [index, time.monotonic()]\n                    except (EOFError, OSError):\n                        # Death can race is_alive(): this job was not accepted.\n                        pending.appendleft((index, code))\n                        stop(slot)\n                        slots[number] = start()\n            active_pipes = [slot[1] for slot in slots if slot[2] is not None]\n            if active_pipes:\n                wait(active_pipes, timeout=.002)  # Wake immediately, no per-program 1ms polling tax.\n            else:\n                time.sleep(.002)\n    finally:\n        for slot in slots:\n            stop(slot)\n\n\nif __name__ == \'__main__\':\n    # Independent supervisor kills the process group on parent death (Linux).\n    if sys.platform.startswith(\'linux\'):\n        import ctypes\n        parent = os.getppid()\n        signal.signal(signal.SIGTERM, lambda *_: os.killpg(os.getpgrp(), signal.SIGKILL))\n        assert ctypes.CDLL(None).prctl(1, signal.SIGTERM, 0, 0, 0) == 0\n        if parent == 1 or os.getppid() != parent:\n            os.killpg(os.getpgrp(), signal.SIGKILL)\n    serve(int(sys.argv[1]))\n', 'final_executor.py': '"""Incremental global execution cache, usable while the GPU generates next batch."""\nimport json\nimport os\nimport signal\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\n\n\nclass Executor:\n    def __init__(self, workers=None, service_path=None):\n        self.workers = workers if workers is not None else (os.cpu_count() or 1)\n        env = dict(os.environ, OMP_NUM_THREADS=\'1\', MKL_NUM_THREADS=\'1\', OPENBLAS_NUM_THREADS=\'1\',\n                   PYTHONHASHSEED=\'0\')\n        self.child = subprocess.Popen([sys.executable, str(service_path or Path(__file__).with_name(\'final_execute.py\')),\n                                      str(self.workers)], stdin=subprocess.PIPE, stdout=subprocess.PIPE,\n                                      stderr=subprocess.DEVNULL, text=True, bufsize=1, env=env,\n                                      start_new_session=True)\n        self.codes = {}\n        self.results = {}\n        self.requests = 0\n        self.started = time.monotonic()\n        self.error = None\n        self.task_wall_seconds = self.task_cpu_seconds = 0.\n        self.first_submit = self.last_result = None\n        def read():\n            try:\n                for line in self.child.stdout:\n                    index, value, *metadata = json.loads(line)\n                    if metadata and metadata[0]:\n                        self.task_wall_seconds += metadata[0][\'task_wall_seconds\']\n                        self.task_cpu_seconds += metadata[0][\'task_cpu_seconds\']\n                    self.last_result = time.monotonic()\n                    self.results[index] = value\n            except Exception as error:\n                self.error = type(error).__name__\n        self.reader = threading.Thread(target=read, daemon=True)\n        self.reader.start()\n\n    def submit(self, candidates):\n        jobs = []\n        for candidate in candidates:\n            self.requests += 1\n            code = candidate[\'code\']\n            if code not in self.codes:\n                index = len(self.codes)\n                self.codes[code] = index\n                jobs.append([index, code])\n        if jobs:\n            if self.first_submit is None:\n                self.first_submit = time.monotonic()\n            self.child.stdin.write(json.dumps(jobs) + \'\\n\')\n            self.child.stdin.flush()\n\n    def attach(self, candidates):\n        for candidate in candidates:\n            index = self.codes[candidate[\'code\']]\n            candidate[\'execution\'] = self.results.get(index, {\'status\': \'budget_not_evaluated\'})\n\n    def drain(self, deadline):\n        while len(self.results) < len(self.codes) and time.monotonic() < deadline:\n            exit_code = self.child.poll()\n            if exit_code is not None:\n                self.error = self.error or \'executor_exit_\'+str(exit_code)\n                break\n            if self.error:\n                break\n            time.sleep(.002)\n        return len(self.results) == len(self.codes)\n\n    def summary(self):\n        return {\'workers\': self.workers, \'requested_candidates\': self.requests,\n                \'unique_programs\': len(self.codes), \'deduplicated\': self.requests-len(self.codes),\n                \'completed_unique\': len(self.results), \'service_wall_seconds\': time.monotonic()-self.started,\n                \'sum_worker_task_wall_seconds\': self.task_wall_seconds,\n                \'sum_worker_task_cpu_seconds\': self.task_cpu_seconds,\n                \'first_submit_to_last_result_seconds\': (self.last_result-self.first_submit\n                    if self.last_result is not None and self.first_submit is not None else None),\n                \'service_wall_includes_GPU_wait\': True,\n                \'wall_limit\': 3., \'cpu_limit\': 2., \'memory_bytes\': 768*1024**2,\n                \'overlap\': True, \'isolation\': \'fresh numeric namespaces / pristine SymPy fork\', \'error\': self.error}\n\n    def close(self):\n        # Never pool.join() unbounded in the CUDA process.\n        try:\n            os.killpg(self.child.pid, signal.SIGKILL)\n        except ProcessLookupError:\n            pass\n        try:\n            self.child.wait(timeout=.1)\n        except subprocess.TimeoutExpired:\n            pass\n        self.child.stdin.close()\n        self.reader.join(timeout=.1)\n        self.child.stdout.close()\n', 'final_inference.py': '"""V7: ordered 2–4 model adaptive ensemble inference on one T4; no targets in decisions/selection."""\nimport os\nimport time\nPROCESS_START = float(os.environ.get(\'FINAL_PROCESS_START\', time.monotonic()))\nimport sys\nimport signal\n# Arm before Torch import: session watchdog can kill the supervisor and its entire tree.\nif __name__ == \'__main__\' and sys.platform.startswith(\'linux\'):\n    import ctypes\n    _parent = os.getppid()\n    signal.signal(signal.SIGTERM, lambda *_: os.killpg(os.getpgrp(), signal.SIGKILL))\n    assert ctypes.CDLL(None).prctl(1, signal.SIGTERM, 0, 0, 0) == 0\n    if _parent == 1 or os.getppid() != _parent:\n        os.killpg(os.getpgrp(), signal.SIGKILL)\nimport argparse\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport torch\nfrom torch import nn\nimport final_core as C\nimport final_fast as Fast\nimport final_speed as Speed\nfrom dataclasses import asdict\n\nDECODE_OPTIONS = Speed.Options(share_cross=True, async_eos=True, eos_every=32,\n    compact_done=True, race_sampler=True, lean_greedy=True)\nfrom final_representation import tokenizer, restore_compact\nfrom final_executor_v6 import Executor, verify_winners\nfrom final_vote import write_answers\nfrom final_adaptive import question_batch\nfrom final_ensemble import parameters, candidates, stage1_decision, stage1_leader_fraction, select_available\n\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(4*1024**2), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef write(path, value):\n    temporary = Path(path).with_suffix(\'.tmp\')\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2)+\'\\n\')\n    temporary.replace(path)\n\n\ndef load(path, device):\n    payload = torch.load(path, map_location=\'cpu\', weights_only=True)\n    assert payload[\'format\'] == \'nlp-hw02-model-only-v1\' and payload[\'optimizer_included\'] is False\n    model = C.SimpleTransformer(**payload[\'model_spec\'], dropout=0.).eval()\n    if device.type == \'cuda\':\n        model.half()\n    model.load_state_dict(payload[\'model\'], strict=True)\n    return model.to(device)\n\n\ndef pack(ids, lps, table, greedy=False):\n    groups = []\n    for sequences, means in zip(ids.tolist(), lps.tolist()):\n        group = []\n        for tokens, lp in zip(sequences, means):\n            active = C.active_tokens(tokens)\n            represented = C.decode_bytes(table, active).decode(\'utf8\', errors=\'replace\')\n            try:\n                code, error = restore_compact(represented), None\n            except (ValueError, IndexError):\n                code, error = \'\', \'invalid representation\'\n            group.append({\'code\': code, \'mean_logprob\': 0. if greedy else (lp if math.isfinite(lp) else None),\n                          \'represented_code\': represented, \'restore_error\': error, \'length\': len(active)})\n        groups.append(group)\n    return groups\n\n\ndef run(config):\n    model_specs, unanimity_threshold = parameters(config)\n    names = [m[\'name\'] for m in model_specs]\n    k1 = {m[\'name\']: m[\'K1\'] for m in model_specs}\n    k2 = {m[\'name\']: m[\'K2\'] for m in model_specs}\n    lanes = config.get(\'sequence_budget\', config.get(\'sample_lanes\', 1024))\n    stage1_batch = {name: question_batch(lanes, k1[name]) for name in names}\n    extra = {name: k2[name]-k1[name] for name in names}\n    extra_batch = question_batch(lanes, max(extra.values())) if any(extra.values()) else 0\n    greedy_lanes = config.get(\'greedy_sequence_budget\', config.get(\'greedy_batch\', lanes))\n    greedy_batch = question_batch(greedy_lanes, 1)\n    scorer = None\n    if config.get(\'scorer_enabled\', False):\n        import importlib\n        module, attribute = config[\'scorer_callable\'].split(\':\')\n        scorer = getattr(importlib.import_module(module), attribute)\n    phases = {\'imports_and_launcher\': time.monotonic()-PROCESS_START}\n    out = Path(config[\'output\']); out.mkdir(parents=True, exist_ok=True)\n    budget = min(180., float(config.get(\'budget_seconds\', 180.)))\n    if not math.isfinite(budget) or budget <= 0:\n        raise ValueError(\'Require a positive finite budget (maximum 180 seconds)\')\n    # GPU/CPU work stops well before the external hard stop, leaving voting/write reserve.\n    work_deadline = PROCESS_START + budget - min(20., budget*.12)\n    execution_deadline = PROCESS_START + budget - min(14., budget*.08)\n    verification_deadline = PROCESS_START + budget - min(10., budget*.06)\n    greedy_deadline = PROCESS_START + budget - min(50., budget*.28)\n    started = time.monotonic()\n    rows = json.loads(Path(config[\'rows\']).read_text())\n    phases[\'read_input\'] = time.monotonic()-started\n    assert rows and all(isinstance(r[\'question\'], str) for r in rows)\n    if config.get(\'expected_rows\') is not None:\n        assert len(rows) == config[\'expected_rows\']\n    device = torch.device(config.get(\'device\', \'cuda:0\'))\n    if device.type == \'cpu\' and config.get(\'cpu_test_expire_sampling\'):\n        work_deadline = PROCESS_START  # Deterministic CPU-only guard fixture.\n    torch.set_num_threads(1)\n    torch.set_num_interop_threads(1)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.use_deterministic_algorithms(True)\n    # The decoder writes each KV slot before reading it; avoid filling unused slots.\n    torch.utils.deterministic.fill_uninitialized_memory = False\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    if device.type == \'cuda\':\n        assert os.environ.get(\'CUDA_VISIBLE_DEVICES\') == \'0\' and torch.cuda.device_count() == 1\n        assert \'T4\' in torch.cuda.get_device_name(0)\n        assert str(torch.__version__) == \'2.11.0+cu128\' and torch.version.cuda == \'12.8\', \'Pinned L2 runtime required\'\n    started = time.monotonic()\n    model_hashes = {m[\'name\']: sha(m[\'path\']) for m in model_specs if Path(m[\'path\']).is_file()}\n    if config.get(\'verify_weights\', True):\n        for m in model_specs:\n            assert model_hashes[m[\'name\']] == m[\'sha256\'], m[\'name\']\n    if config.get(\'tokenizer_sha\'):\n        assert sha(config[\'tokenizer\']) == config[\'tokenizer_sha\']\n    models = {m[\'name\']: load(m[\'path\'], device) for m in model_specs}\n    model_bytes = {name: sum(t.numel()*t.element_size() for t in model.state_dict().values())\n                   for name, model in models.items()}\n    model_specs_loaded = {name: {\'width\': model.width, \'decoder_layers\': len(model.transformer.decoder.layers)}\n                          for name, model in models.items()}\n    tok, table = tokenizer(config[\'tokenizer\'], 30)\n    encoded = [[C.BOS]+C.encode(tok, r[\'question\'])+[C.EOS] for r in rows]\n    phases[\'load_models_tokenizer_encode\'] = time.monotonic()-started\n    started = time.monotonic()\n    # No stochastic warmup; seed is reset separately for each model afterwards.\n    for model in models.values():\n        src = torch.tensor([encoded[0]], device=device)\n        ctx = Fast.Context(model, src, 16)\n        Speed.decode(ctx, options=DECODE_OPTIONS)\n        del ctx, src\n    C.sync(device)\n    phases[\'warmup\'] = time.monotonic()-started\n    generation_workers = 2\n    tail_workers = max(2, min(4, os.cpu_count() or 1))\n    executor = Executor(generation_workers)\n    records = [{\'greedy\': None, \'samples\': {name: [] for name in names}} for _ in rows]\n    guard_events = []\n    cap = int(config.get(\'cap\', 512))\n    restore_seconds = 0.\n    attempts = []\n    batch_stats = []\n    stopped_rows, escalated_rows = [], []\n    stage2_waves = []\n    model_rng = {}\n    effective_batches = {}\n    stage2_order = []\n    leader_fractions = {}\n\n    def generate(name, order, batch, k, temperature, target, deadline, stage=0):\n        nonlocal restore_seconds\n        batch = min(batch, effective_batches.get((name, k), batch))\n        offset = 0\n        while offset < len(order):\n            C.check_deadline(deadline)\n            selected = order[offset:offset+batch]\n            src = ctx = seq = means = None\n            rng_cpu = torch.get_rng_state() if temperature is not None else None\n            rng_gpu = torch.cuda.get_rng_state(device) if temperature is not None and device.type == \'cuda\' else None\n            had_oom = False\n            batch_started = time.monotonic()\n            try:\n                src = nn.utils.rnn.pad_sequence([torch.tensor(encoded[i]) for i in selected],\n                    batch_first=True, padding_value=C.PAD).to(device)\n                ctx = Fast.Context(models[name], src, cap)\n                decode_stats = {}\n                seq, means, _ = Speed.decode(ctx, k, temperature, deadline=deadline,\n                    compute_logprob=temperature is not None, options=DECODE_OPTIONS, stats=decode_stats)\n                begin = time.monotonic()\n                groups = pack(seq, means, table, greedy=temperature is None)\n                restore_seconds += time.monotonic()-begin\n                for i, group in zip(selected, groups):\n                    if target == \'greedy\':\n                        records[i][target] = group[0]\n                    else:\n                        records[i][\'samples\'][target].extend(group)\n                executor.submit([c for group in groups for c in group])\n                batch_stats.append({\'phase\': target, \'stage\': stage, \'questions\': len(selected), \'sequences\': len(selected)*k,\n                                    \'row_ids\': selected, \'source_tokens\': src.shape[1], \'decode_steps\': seq.shape[-1],\n                                    \'decoder\': decode_stats, \'wall_seconds\': time.monotonic()-batch_started})\n                offset += len(selected)\n                if (device.type == \'cpu\' and target != \'greedy\' and\n                    config.get(\'cpu_test_expire_after_sample_batches\') ==\n                    sum(s[\'phase\'] != \'greedy\' for s in batch_stats)):\n                    raise C.BudgetExpired(\'CPU partial-candidate guard fixture\')\n            except torch.cuda.OutOfMemoryError:\n                had_oom = True\n                if rng_cpu is not None:\n                    torch.set_rng_state(rng_cpu)\n                if rng_gpu is not None:\n                    torch.cuda.set_rng_state(rng_gpu, device)\n                attempts.append({\'model\': name, \'offset\': offset, \'question_batch\': batch, \'K\': k})\n                batch //= 2\n                if batch < 1:\n                    raise\n                effective_batches[name, k] = batch\n            finally:\n                del src, ctx, seq, means\n                if had_oom:\n                    # Only OOM requires collection. Normal batches reuse the CUDA allocator.\n                    import gc\n                    gc.collect()\n                    if device.type == \'cuda\':\n                        torch.cuda.empty_cache()\n\n    try:\n        started = time.monotonic()\n        greedy_order = sorted(range(len(rows)), key=lambda i: (len(encoded[i]), i))\n        generate(names[0], greedy_order, greedy_batch, 1, None, \'greedy\', greedy_deadline)\n        phases[\'greedy_generation_restore_submit\'] = time.monotonic()-started\n        # Completed before sampling: the independent supervisor can publish it after a hard kill.\n        started = time.monotonic()\n        write_answers(out/\'greedy-fallback.jsonl\', [r[\'greedy\'][\'code\'] for r in records])\n        write(out/\'greedy-ready.json\', {\'rows\': len(rows), \'seconds\': time.monotonic()-PROCESS_START})\n        phases[\'greedy_checkpoint_write\'] = time.monotonic()-started\n        order = greedy_order\n        for name in names:\n            target = name\n            started = time.monotonic()\n            torch.manual_seed(270042)\n            if device.type == \'cuda\':\n                torch.cuda.manual_seed_all(270042)\n            try:\n                generate(name, order, stage1_batch[name], k1[name], .7, target, work_deadline, stage=1)\n            except C.BudgetExpired:\n                guard_events.append({\'phase\': target, \'elapsed_seconds\': time.monotonic()-PROCESS_START})\n            model_rng[name] = (torch.get_rng_state(),\n                              torch.cuda.get_rng_state(device) if device.type == \'cuda\' else None)\n            phases[\'stage1_\'+target+\'_sampling_restore_submit\'] = time.monotonic()-started\n            if guard_events:\n                break\n        # Global priority requires all stage-one evidence, including the slowest\n        # CPU tasks. Execution still overlaps greedy and all stage-one GPU phases.\n        pending = set(order)\n        phases[\'stage1_decision_wait\'] = 0.\n        phases[\'stage2_sampling_restore_submit\'] = 0.\n        try:\n            if not guard_events:\n                begin = time.monotonic()\n                try:\n                    if not executor.drain(work_deadline):\n                        raise C.BudgetExpired(\'Stage-one executor unavailable\')\n                    for i in order:\n                        C.check_deadline(work_deadline)\n                        record = records[i]\n                        executor.attach(candidates(record, model_specs, \'K1\'))\n                        decision = stage1_decision(record, model_specs, unanimity_threshold)\n                        if decision is None:\n                            raise C.BudgetExpired(\'Incomplete stage-one execution\')\n                        pending.remove(i)\n                        if decision:\n                            stopped_rows.append(i)\n                        else:\n                            escalated_rows.append(i)\n                            leader_fractions[i] = stage1_leader_fraction(record, model_specs)\n                    stage2_order = sorted(escalated_rows, key=lambda i: (leader_fractions[i], len(encoded[i]), i))\n                finally:\n                    phases[\'stage1_decision_wait\'] += time.monotonic()-begin\n                # Complete each model for a priority batch before touching the\n                # next one. The largest per-model extra sets the shared wave size.\n                for offset in range(0, len(stage2_order), extra_batch) if extra_batch else []:\n                    C.check_deadline(work_deadline)\n                    ready = stage2_order[offset:offset+extra_batch]\n                    stage2_waves.append({\'questions\': len(ready), \'pending_stage1_rows\': 0,\n                        \'row_ids\': ready, \'leader_fraction_min\': leader_fractions[ready[0]],\n                        \'leader_fraction_max\': leader_fractions[ready[-1]]})\n                    begin = time.monotonic()\n                    try:\n                        for name in names:\n                            target = name\n                            if not extra[name]:\n                                continue\n                            cpu_rng, cuda_rng = model_rng[name]\n                            torch.set_rng_state(cpu_rng)\n                            if cuda_rng is not None:\n                                torch.cuda.set_rng_state(cuda_rng, device)\n                            generate(name, ready, extra_batch, extra[name], .7, target, work_deadline, stage=2)\n                            model_rng[name] = (torch.get_rng_state(),\n                                torch.cuda.get_rng_state(device) if device.type == \'cuda\' else None)\n                    finally:\n                        phases[\'stage2_sampling_restore_submit\'] += time.monotonic()-begin\n        except C.BudgetExpired:\n            guard_events.append({\'phase\': \'adaptive_stage2\', \'elapsed_seconds\': time.monotonic()-PROCESS_START})\n        started = time.monotonic()\n        # Preserve the same queue/cache and in-flight tasks; only add idle slots.\n        executor.resize(tail_workers)\n        drained = executor.drain(execution_deadline)\n        phases[\'execution_tail_wait\'] = time.monotonic()-started\n        if not drained:\n            guard_events.append({\'phase\': \'execution\', \'elapsed_seconds\': time.monotonic()-PROCESS_START})\n        all_generated = [c for r in records for c in candidates(r, model_specs)]\n        executor.attach(all_generated)\n        execution = executor.summary()\n    finally:\n        started = time.monotonic()\n        executor.close()\n        phases[\'executor_shutdown\'] = time.monotonic()-started\n    started = time.monotonic()\n    winners = []\n    fallback = []\n    partial_vote_rows = []\n    executed_sample_counts = []\n    for i, r in enumerate(records):\n        try:\n            winner, count = select_available(r, model_specs, question=rows[i][\'question\'], scorer=scorer,\n                score_weight=float(config.get(\'scorer_weight\', 0.)), deadline=verification_deadline)\n        except C.BudgetExpired:\n            winner, count = select_available(r, model_specs)\n            guard_events.append({\'phase\': \'scorer\', \'row\': i})\n        winners.append(winner)\n        executed_sample_counts.append(count)\n        if count == 0:\n            fallback.append(i)\n        elif count < sum(m[\'K1\' if i in stopped_rows else \'K2\'] for m in model_specs):\n            partial_vote_rows.append(i)\n    phases[\'vote\'] = time.monotonic()-started\n    started = time.monotonic()\n    verification = verify_winners(winners, verification_deadline, tail_workers)\n    for i in verification[\'mismatch_rows\']:\n        winners[i] = records[i][\'greedy\']\n        if i not in fallback:\n            fallback.append(i)\n    if verification[\'mismatch_rows\']:\n        guard_events.append({\'phase\': \'winner_reexecution\', \'rows\': verification[\'mismatch_rows\'],\n                             \'elapsed_seconds\': time.monotonic()-PROCESS_START})\n    phases[\'winner_reexecution\'] = time.monotonic()-started\n    started = time.monotonic()\n    write_answers(out/\'answers.jsonl\', [w[\'code\'] for w in winners])\n    closed = time.monotonic()\n    phases[\'answer_write\'] = closed-started\n    phases[\'bookkeeping_and_executor_start\'] = closed-PROCESS_START-sum(phases.values())\n    timing = {\'full_pipeline_seconds\': closed-PROCESS_START, \'rows\': len(rows), \'phases_seconds\': phases,\n              \'restore_seconds_in_generation\': restore_seconds, \'execution\': execution,\n              \'deadline_guard\': {\'triggered\': bool(guard_events), \'events\': guard_events,\n                                 \'fallback_rows\': sorted(fallback), \'fallback_count\': len(fallback)},\n              \'winner_reexecution\': verification,\n              \'no_numeric_winner_rows\': [i for i, w in enumerate(winners) if w[\'execution\'][\'status\'] != \'numeric\'],\n              \'input_sha256\': sha(config[\'rows\']),\n              \'question_order_sha256\': hashlib.sha256(json.dumps(\n                  [r[\'question\'] for r in rows], ensure_ascii=False).encode()).hexdigest(),\n              \'answers_sha256\': sha(out/\'answers.jsonl\'),\n              \'OOM_attempts\': attempts, \'single_gpu\': device.type == \'cuda\', \'seed\': 270042,\n              \'adaptive\': {\'K1\': k1, \'K2\': k2, \'unanimity_threshold\': unanimity_threshold,\n                           \'stopped_rows\': sorted(stopped_rows), \'escalated_rows\': sorted(escalated_rows),\n                           \'undecided_rows\': sorted(pending), \'stage2_waves\': stage2_waves,\n                           \'stage2_order\': stage2_order, \'stage2_priority\': \'stage1_leader_fraction_then_length_then_id\',\n                           \'stage1_leader_fractions\': leader_fractions,\n                           \'mean_sample_count\': sum(len(v) for r in records for v in r[\'samples\'].values())/len(rows),\n                           \'scorer_enabled\': config.get(\'scorer_enabled\', False)},\n              \'greedy_batch\': greedy_batch, \'greedy_sequence_budget\': greedy_lanes,\n              \'sample_lanes\': lanes, \'sequence_budget\': lanes,\n              \'length_bucketing\': True, \'generation_batches\': batch_stats,\n              \'decoder_options\': asdict(DECODE_OPTIONS),\n              \'executor_policy\': {\'generation_workers\': generation_workers, \'tail_workers\': tail_workers,\n                                  \'nice_increment\': 10, \'resize_preserves_queue_and_cache\': True},\n              \'model_sha256\': model_hashes, \'model_order\': names,\n              \'resident_model_bytes\': model_bytes, \'resident_model_total_bytes\': sum(model_bytes.values()),\n              \'model_specs\': model_specs_loaded,\n              \'mean_samples_per_model\': {name: sum(len(r[\'samples\'][name]) for r in records)/len(rows) for name in names},\n              \'total_device_memory_bytes\': torch.cuda.get_device_properties(device).total_memory if device.type == \'cuda\' else None,\n              \'peak_reserved_bytes\': torch.cuda.max_memory_reserved(device) if device.type == \'cuda\' else None,\n              \'peak_allocated_bytes\': torch.cuda.max_memory_allocated(device) if device.type == \'cuda\' else None,\n              \'deterministic_algorithms\': True, \'python_hash_seed\': os.environ.get(\'PYTHONHASHSEED\'),\n              \'timer\': \'launcher process start through answers file close; imports/load/warmup/exec/vote/write included\'}\n    timing[\'deadline_guard\'].update(partial_vote_rows=partial_vote_rows,\n        partial_vote_count=len(partial_vote_rows), no_executed_sample_count=executed_sample_counts.count(0),\n        executed_sample_counts=executed_sample_counts,\n        generation_stop_seconds=work_deadline-PROCESS_START, execution_stop_seconds=execution_deadline-PROCESS_START,\n        verification_stop_seconds=verification_deadline-PROCESS_START)\n    write(out/\'timing.json\', timing)\n    # Diagnostic outputs and gold scoring are outside the answer timer, no gold is passed to select.\n    if config.get(\'reference\'):\n        reference = json.loads(Path(config[\'reference\']).read_text())\n        assert len(reference) == len(winners)\n        matches = [i for i, (ref, win) in enumerate(zip(reference, winners)) if\n                   ref[\'status\'] == win[\'execution\'][\'status\'] == \'numeric\' and math.isclose(\n                       ref[\'value\'], win[\'execution\'][\'value\'], rel_tol=1e-6, abs_tol=1e-6)]\n        write(out/\'quality.json\', {\'matches\': len(matches), \'n\': len(winners), \'accuracy_pp\': 100*len(matches)/len(winners),\n                                 \'match_indices\': matches, \'expected_matches\': config.get(\'expected_matches\', 1511),\n                                 \'delta_pp\': 100*(len(matches)-config.get(\'expected_matches\', 1511))/len(winners),\n                                 \'tolerance_pp\': 1.,\n                                 \'passed\': abs(100*(len(matches)-config.get(\'expected_matches\', 1511))/len(winners)) <= 1. and not guard_events})\n        import gzip\n        with gzip.open(out/\'candidate-journal.json.gz\', \'wt\', encoding=\'utf8\') as stream:\n            json.dump(records, stream, ensure_ascii=False)\n    return timing\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'config\')\n    run(json.loads(Path(parser.parse_args().config).read_text()))\n', 'final_runner.py': '"""Torch-free process supervisor. Hard stop and greedy publication share the honest timer."""\nimport time\nimport os\nSTART = float(os.environ.get(\'FINAL_PROCESS_START\', time.monotonic()))\nimport json\nimport signal\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nif __name__ == \'__main__\' and sys.platform.startswith(\'linux\'):\n    import ctypes\n    _parent = os.getppid()\n    signal.signal(signal.SIGTERM, lambda *_: os.killpg(os.getpgrp(), signal.SIGKILL))\n    assert ctypes.CDLL(None).prctl(1, signal.SIGTERM, 0, 0, 0) == 0\n    if _parent == 1 or os.getppid() != _parent:\n        os.killpg(os.getpgrp(), signal.SIGKILL)\n\n\ndef supervise(config_file, start=None):\n    start = START if start is None else start\n    config = json.loads(Path(config_file).read_text())\n    out = Path(config[\'output\']); out.mkdir(parents=True, exist_ok=True)\n    old = [name for name in (\'answers.jsonl\', \'greedy-fallback.jsonl\', \'greedy-ready.json\',\n                             \'timing.json\', \'summary.json\', \'worker.log\') if (out/name).exists()]\n    if old:\n        raise ValueError(\'Fresh cold run requires a new output directory; existing artifacts: \'+\',\'.join(old))\n    budget = min(180., float(config.get(\'budget_seconds\', 180.)))\n    env = dict(os.environ, CUDA_VISIBLE_DEVICES=\'0\', FINAL_PROCESS_START=str(start),\n               TOKENIZERS_PARALLELISM=\'false\', OMP_NUM_THREADS=\'1\', MKL_NUM_THREADS=\'1\', OPENBLAS_NUM_THREADS=\'1\',\n               PYTHONHASHSEED=\'0\', CUBLAS_WORKSPACE_CONFIG=\':4096:8\')\n    child = None\n    stopped = False\n    with (out/\'worker.log\').open(\'w\') as log:\n        child = subprocess.Popen([sys.executable, str(Path(__file__).with_name(\'final_inference.py\')), str(config_file)],\n                                 env=env, start_new_session=True, stdout=log, stderr=log)\n        try:\n            while child.poll() is None:\n                if time.monotonic() >= start+budget-min(8., budget*.045):\n                    stopped = True\n                    break\n                time.sleep(.01)\n        finally:\n            # Executor is a separate session; its Linux parent-death guard kills the tree.\n            try:\n                os.killpg(child.pid, signal.SIGKILL)\n            except ProcessLookupError:\n                pass\n            try:\n                child.wait(timeout=.1)\n            except subprocess.TimeoutExpired:\n                pass\n    timer_path = out/\'timing.json\'\n    if stopped or child.returncode or not timer_path.exists():\n        # Fatal GPU/driver/init failure before greedy completion cannot produce real greedy.\n        # Fail explicitly, never fabricate code or silently claim the time gate passed.\n        ready = out/\'greedy-ready.json\'\n        if not ready.exists():\n            report = {\'status\': \'failed_before_greedy\', \'elapsed_seconds\': time.monotonic()-start,\n                      \'exit_code\': child.returncode, \'hard_stop\': stopped}\n            (out/\'summary.json\').write_text(json.dumps(report, indent=2)+\'\\n\')\n            raise RuntimeError(\'Cold pipeline failed before greedy; inspect worker.log\')\n        data = (out/\'greedy-fallback.jsonl\').read_bytes()\n        with (out/\'answers.jsonl\').open(\'wb\') as stream:\n            stream.write(data)\n        closed = time.monotonic()\n        timing = {\'full_pipeline_seconds\': closed-start, \'rows\': json.loads(ready.read_text())[\'rows\'],\n                  \'deadline_guard\': {\'triggered\': True, \'supervisor\': True, \'fallback_rows\': \'all\',\n                                     \'fallback_count\': json.loads(ready.read_text())[\'rows\']},\n                  \'phases_seconds\': {}, \'worker_exit_code\': child.returncode, \'hard_stop\': stopped}\n        timer_path.write_text(json.dumps(timing, indent=2)+\'\\n\')\n    else:\n        timing = json.loads(timer_path.read_text())\n    timing[\'status\'] = \'guarded\' if timing[\'deadline_guard\'][\'triggered\'] else \'complete\'\n    timing[\'under_180_seconds\'] = timing[\'full_pipeline_seconds\'] < 180.\n    timing[\'target_150_seconds\'] = timing[\'full_pipeline_seconds\'] <= 150.\n    timing[\'target_155_seconds\'] = timing[\'full_pipeline_seconds\'] <= 155.\n    timing[\'target_120_seconds\'] = timing[\'full_pipeline_seconds\'] <= 120.\n    (out/\'summary.json\').write_text(json.dumps(timing, indent=2)+\'\\n\')\n    assert timing[\'full_pipeline_seconds\'] < budget, \'Budget exceeded: reject this run\'\n    return timing\n\n\nif __name__ == \'__main__\':\n    report = supervise(sys.argv[1])\n    print(json.dumps({k: report[k] for k in [\'status\', \'rows\', \'full_pipeline_seconds\']}, separators=(\',\', \':\')))\n', 'final_session.py': '"""V8: one cold test and tune, ordered dataset/model config and independent watchdog (no Kaggle API/secrets)."""\nimport gzip\nimport hashlib\nimport json\nimport math\nimport os\nimport signal\nimport subprocess\nimport sys\nimport threading\nimport time\nimport zipfile\nfrom pathlib import Path\nfrom final_split import fixed_split\nfrom final_vote import validate_answers\nfrom final_ensemble import parameters\n\nINPUT_SHA = {\n    \'tokenizer.json\': \'261f25b5d4be006a6a00e41e0eafbe522580741da627ba670f6b9805a2659b7d\',\n    \'test.json\': \'20cacb334e3fc81291433d2173d9085b17a15e1834bd84aea67b350403182503\',\n    \'heldout4k.json\': \'28a3c0c9088c5b4a17d75d216e4abc4262cc9ca62884966c43e7ec09b6f040c2\',\n    \'heldout4k-reference-execution.json\': \'3855680e9a37292aa2e351b8e8e4f7a5cbbe76444096d136470e4a9be8797111\'}\n\n\ndef write(path, value):\n    temp = Path(path).with_suffix(\'.tmp\')\n    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2)+\'\\n\')\n    temp.replace(path)\n\n\nclass Watchdog:\n    def __init__(self, output, start):\n        self.output, self.start = Path(output), start\n        self.children = set()\n        self.done = threading.Event()\n        self.triggered = False\n\n    def kill(self):\n        for pid in list(self.children):\n            try:\n                os.killpg(pid, signal.SIGKILL)\n            except ProcessLookupError:\n                pass\n\n    def arm(self):\n        def watch():\n            while not self.done.wait(.1):\n                elapsed = time.monotonic()-self.start\n                if elapsed >= 660. and not self.triggered:\n                    self.triggered = True\n                    self.kill()\n                    write(self.output/\'watchdog.json\', {\'triggered\': True, \'elapsed_seconds\': elapsed})\n                if elapsed >= 690.:\n                    self.kill()\n                    os._exit(0)\n        threading.Thread(target=watch, daemon=True, name=\'final-session-watchdog\').start()\n        return self\n\n    def finish(self):\n        self.kill()\n        self.done.set()\n\n\ndef prepare_inputs(root, target, benchmark=True):\n    """Extract only 4 small JSONs from starter ZIP; never unpack initial/control/train weights."""\n    root, target = Path(root), Path(target)\n    target.mkdir(parents=True, exist_ok=True)\n    for name, expected in INPUT_SHA.items():\n        if not benchmark and name not in (\'tokenizer.json\', \'test.json\'):\n            continue\n        direct = list(root.rglob(name))\n        if len(direct) == 1:\n            path = direct[0]\n        elif not direct:\n            archives = list(root.rglob(\'26b-training-inputs-d0-initial-and-evaluation.zip\'))\n            assert len(archives) == 1, \'Expected direct starter files or one known starter ZIP\'\n            with zipfile.ZipFile(archives[0]) as archive:\n                members = [m for m in archive.namelist() if Path(m).name == name]\n                assert len(members) == 1\n                path = target/name\n                with path.open(\'wb\') as stream:\n                    stream.write(archive.read(members[0]))\n        else:\n            raise ValueError(\'Ambiguous input \'+name)\n        assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, name\n        yield name, str(path)\n\n\ndef unique(root, name):\n    paths = list(Path(root).rglob(name))\n    assert len(paths) == 1, \'Missing or ambiguous \'+name\n    return str(paths[0])\n\n\ndef weight_path(root, model):\n    if \'relative_path\' not in model:\n        return unique(root, model[\'filename\'])\n    relative = Path(model[\'relative_path\'])\n    assert not relative.is_absolute() and \'..\' not in relative.parts\n    path = Path(root)/relative\n    assert path.is_file() and path.name == model[\'filename\'], \'Missing pinned model \'+str(relative)\n    return str(path)\n\n\ndef suite(inputs_root, weights_roots, output, module_dir, watchdog, benchmark=True, ensemble_config=None):\n    output = Path(output)\n    report = {\'status\': \'running\', \'test\': [], \'GPU_runs\': 0, \'training\': False,\n              \'budget_seconds\': 720, \'internet\': False, \'single_gpu\': True}\n    write(output/\'summary.json\', report)\n    try:\n        inputs = dict(prepare_inputs(inputs_root, output/\'inputs\', benchmark=benchmark))\n        assert ensemble_config is not None, \'Pass the reviewed ensemble config\'\n        model_specs, _ = parameters(ensemble_config)\n        models = [dict(m, path=weight_path(weights_roots[m[\'dataset\']], m)) for m in model_specs]\n        heldout = json.loads(Path(inputs[\'heldout4k.json\']).read_text()) if benchmark else None\n        ids = fixed_split()[\'tune2048\']\n        rows_path, ref_path = output/\'tune2048.json\', output/\'tune2048-reference.json\'\n        if benchmark:\n            references = json.loads(Path(inputs[\'heldout4k-reference-execution.json\']).read_text())\n            refs = references[\'rows\'] if isinstance(references, dict) else references\n            assert len(heldout) == len(refs) == 4096\n            write(rows_path, [{\'question\': heldout[i][\'question\']} for i in ids])\n            write(ref_path, [refs[i] for i in ids])\n            report[\'stored_tune\'] = {\'matches\': ensemble_config[\'expected_matches\'], \'n\': 2048,\n                                     \'origin\': \'local verified K8 replay; new RNG samples can differ\'}\n        base = dict(ensemble_config, models=models, tokenizer=inputs[\'tokenizer.json\'],\n                    tokenizer_sha=INPUT_SHA[\'tokenizer.json\'], workers=2, verify_weights=True,\n                    budget_seconds=180., greedy_batch=1024)\n        def launch(name, rows, expected, **extra):\n            assert not watchdog.triggered and time.monotonic()-watchdog.start < 480.\n            destination = output/name\n            destination.mkdir(exist_ok=True)\n            config = dict(base, rows=str(rows), output=str(destination), expected_rows=expected, **extra)\n            config_path = destination/\'config.json\'; write(config_path, config)\n            env = dict(os.environ, CUDA_VISIBLE_DEVICES=\'0\')\n            log_path = destination/\'supervisor.log\'\n            with log_path.open(\'w\') as log:\n                # Timestamp before creation includes Python interpreter startup and all imports.\n                env[\'FINAL_PROCESS_START\'] = str(time.monotonic())\n                child = subprocess.Popen([sys.executable, str(Path(module_dir)/\'final_runner.py\'), str(config_path)],\n                                         start_new_session=True, env=env, stdout=log, stderr=log)\n                watchdog.children.add(child.pid)\n                try:\n                    until = min(watchdog.start+660., time.monotonic()+config[\'budget_seconds\']+5.)\n                    next_heartbeat = time.monotonic()+30.\n                    while child.poll() is None and time.monotonic() < until and not watchdog.triggered:\n                        if time.monotonic() >= next_heartbeat:\n                            print(name, \'running; session_s\', round(time.monotonic()-watchdog.start), flush=True)\n                            next_heartbeat += 30.\n                        time.sleep(.1)\n                    if child.poll() is None:\n                        os.killpg(child.pid, signal.SIGKILL)\n                    child.wait(timeout=1.)\n                    assert child.returncode == 0, name+\' failed; inspect small worker/supervisor logs\'\n                finally:\n                    watchdog.children.discard(child.pid)\n            validate_answers(destination/\'answers.jsonl\', expected)\n            report[\'GPU_runs\'] += 1\n            return json.loads((destination/\'summary.json\').read_text())\n        for number in range(1, 2):\n            row = launch(f\'test-{number}\', inputs[\'test.json\'], 1919)\n            report[\'test\'].append(row)\n            if number == 1:\n                # The published answer is always the FIRST cold test run, never picked by heldout score.\n                (output/\'answers.jsonl\').write_bytes((output/\'test-1/answers.jsonl\').read_bytes())\n            write(output/\'summary.json\', report)\n            print(\'test\', number, row[\'status\'], round(row[\'full_pipeline_seconds\'], 3), flush=True)\n        if not benchmark:\n            report[\'status\'] = report[\'test\'][0][\'status\']\n            return report\n        report[\'tune\'] = launch(\'tune\', rows_path, 2048, reference=str(ref_path))\n        quality_path = output/\'tune/quality.json\'\n        report[\'quality\'] = json.loads(quality_path.read_text()) if quality_path.exists() else {\'passed\': False, \'reason\': \'supervisor fallback\'}\n        write(output/\'summary.json\', report)\n        digests = [hashlib.sha256((output/f\'test-{i}/answers.jsonl\').read_bytes()).hexdigest()\n                   for i in range(1, 2)]\n        report[\'answers_sha256\'] = digests\n        report[\'deterministic_answers\'] = None  # One cold test; repeatability not measured\n        report[\'gates\'] = {\'test_all_under_180\': all(r[\'under_180_seconds\'] for r in report[\'test\']),\n                           \'test_with_40s_margin\': all(r[\'full_pipeline_seconds\'] <= 140. for r in report[\'test\']),\n                           \'tune_under_180_complete\': report[\'tune\'][\'under_180_seconds\'] and report[\'tune\'][\'status\'] == \'complete\',\n                           \'resident_models_and_T4_memory\': all(\n                               len(r.get(\'model_order\', [])) == len(models) and\n                               r.get(\'peak_reserved_bytes\', float(\'inf\')) < r.get(\'total_device_memory_bytes\', 0) and\n                               r.get(\'peak_allocated_bytes\', float(\'inf\'))+256*1024**2 < r.get(\'total_device_memory_bytes\', 0)\n                               for r in report[\'test\']+[report[\'tune\']]),\n                           \'no_test_fallback\': all(r[\'status\'] == \'complete\' and\n                               r[\'deadline_guard\'][\'fallback_count\'] == 0 for r in report[\'test\']),\n                           \'quality_within_1pp\': report[\'quality\'][\'passed\']}\n        report[\'status\'] = \'passed\' if all(report[\'gates\'].values()) else \'review_required\'\n    except BaseException as error:\n        report.update(status=\'failed\', error_type=type(error).__name__, error=str(error))\n        raise\n    finally:\n        report[\'session_seconds\'] = time.monotonic()-watchdog.start\n        write(output/\'summary.json\', report)\n        watchdog.finish()\n    return report\n', 'final_adaptive.py': '"""CPU-only adaptive decisions and the frozen v3 vote extended to arbitrary K.\n\nK1: stage-one samples PER model. K2: total samples PER escalated model.\nUnanimity is relative to greedy (isclose is intentionally not transitive here).\n"""\nimport math\nfrom final_choose import choose\n\n\ndef parameters(config):\n    k1, k2 = config.get(\'K1\', 2), config.get(\'K2\', 4)\n    threshold = config.get(\'unanimity_threshold\', 1.)\n    if (isinstance(k1, bool) or not isinstance(k1, int) or k1 < 1 or\n        isinstance(k2, bool) or not isinstance(k2, int) or k2 < k1):\n        raise ValueError(\'Require integer 1 <= K1 <= K2 (per model)\')\n    if not isinstance(threshold, (int, float)) or not math.isfinite(threshold) or not 0 < threshold <= 1:\n        raise ValueError(\'Require 0 < unanimity_threshold <= 1\')\n    return k1, k2, float(threshold)\n\n\ndef numeric(candidate):\n    execution = candidate.get(\'execution\', {})\n    value = execution.get(\'value\')\n    return (execution.get(\'status\') == \'numeric\' and isinstance(value, (int, float))\n            and math.isfinite(value))\n\n\ndef same(a, b):\n    return math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-6)\n\n\ndef question_batch(sequence_budget, k):\n    """K lanes per question, including greedy K=1; never exceed the lane budget."""\n    if (isinstance(sequence_budget, bool) or not isinstance(sequence_budget, int) or\n        isinstance(k, bool) or not isinstance(k, int) or k < 1 or sequence_budget < k):\n        raise ValueError(\'Require integer sequence_budget >= K >= 1\')\n    return sequence_budget//k\n\n\ndef stage1_leader_fraction(record, k1=2):\n    """Largest numeric vote component / all stage-one candidates (failures included).\n\n    Use the frozen vote\'s connected-component semantics; failed programs supply\n    no agreement. This is scheduling evidence, never a gold-dependent decision.\n    """\n    if stage1_decision(record, k1) is None:\n        raise ValueError(\'Priority requires completed stage-one executions\')\n    candidates = [record[\'greedy\']]+record[\'main\'][:k1]+record[\'long\'][:k1]\n    remaining = [c for c in candidates if numeric(c)]\n    largest = 0\n    while remaining:\n        component = [remaining.pop(0)]\n        while True:\n            adjacent = [c for c in remaining if any(same(c[\'execution\'][\'value\'],\n                        v[\'execution\'][\'value\']) for v in component)]\n            if not adjacent:\n                break\n            component.extend(adjacent)\n            identities = {id(c) for c in adjacent}\n            remaining = [c for c in remaining if id(c) not in identities]\n        largest = max(largest, len(component))\n    return largest/len(candidates)\n\n\ndef stage1_decision(record, k1=2, threshold=1.):\n    """None: await execution; True: stop; False: escalate. No targets/scores."""\n    if record[\'greedy\'] is None or any(len(record[key]) < k1 for key in (\'main\', \'long\')):\n        return None\n    candidates = [record[\'greedy\']] + record[\'main\'][:k1] + record[\'long\'][:k1]\n    if any(c.get(\'execution\', {}).get(\'status\', \'budget_not_evaluated\') == \'budget_not_evaluated\'\n           for c in candidates):\n        return None\n    # A failed or nonfinite program never supplies evidence for early stopping.\n    if not all(numeric(c) for c in candidates):\n        return False\n    value = candidates[0][\'execution\'][\'value\']\n    return sum(same(c[\'execution\'][\'value\'], value) for c in candidates)/len(candidates) >= threshold\n\n\ndef select(greedy, main, long, k=8, question=None, scorer=None, score_weight=0., deadline=None):\n    """count -> greedy -> mean LP -> stable order; greedy code in winning component.\n\n    Optional score(question, represented_code) adds weight * max(score) to the\n    cluster\'s vote count. Disabled callers never invoke/import a reviewer.\n    """\n    samples = [c for c in main[:k]+long[:k] if c.get(\'mean_logprob\') is not None\n               and math.isfinite(c[\'mean_logprob\'])]\n    winner = choose({\'greedy\': greedy, \'samples\': {\'0.7\': samples}},\n                    {\'mode\': \'vote\', \'temperature\': .7, \'k\': len(samples), \'greedy_vote\': True})\n    if not numeric(winner):\n        return greedy\n    good = [c for c in [greedy]+samples if numeric(c)]\n    components = []\n    remaining = good[:]\n    while remaining:\n        component = [remaining.pop(0)]\n        while True:\n            adjacent = [c for c in remaining if any(same(c[\'execution\'][\'value\'],\n                        v[\'execution\'][\'value\']) for v in component)]\n            if not adjacent:\n                break\n            component.extend(adjacent)\n            identities = {id(c) for c in adjacent}\n            remaining = [c for c in remaining if id(c) not in identities]\n        components.append(component)\n    if scorer is not None and score_weight:\n        if question is None or not math.isfinite(score_weight):\n            raise ValueError(\'Scorer requires question and finite weight\')\n        cache = {}\n        def key(component):\n            for c in component:\n                if deadline is not None:\n                    from final_core import check_deadline\n                    check_deadline(deadline)\n                represented = c[\'represented_code\']\n                if represented not in cache:\n                    cache[represented] = float(scorer(question, represented))\n                    if not math.isfinite(cache[represented]):\n                        raise ValueError(\'Nonfinite reviewer score\')\n            return (len(component)+score_weight*max(cache[c[\'represented_code\']] for c in component),\n                    any(c is greedy for c in component),\n                    sum(c[\'mean_logprob\'] for c in component)/len(component))\n        component = max(components, key=key)\n        winner = max(component, key=lambda c: c[\'mean_logprob\'])\n    else:\n        component = next(group for group in components if any(c is winner for c in group))\n    return greedy if any(c is greedy for c in component) else winner\n\n\ndef select_available(record, k=8, **kwargs):\n    available = {key: [c for c in record[key] if\n                 c.get(\'execution\', {}).get(\'status\', \'budget_not_evaluated\') != \'budget_not_evaluated\']\n                 for key in (\'main\', \'long\')}\n    count = len(available[\'main\'])+len(available[\'long\'])\n    return select(record[\'greedy\'], available[\'main\'], available[\'long\'], k, **kwargs), count\n\n\ndef replay_record(greedy, main, long, k1=2, k2=8, threshold=1.):\n    """The same stage-one decision and selector as the live worker, in journal order."""\n    record = {\'greedy\': greedy, \'main\': main[:k1], \'long\': long[:k1]}\n    decision = stage1_decision(record, k1, threshold)\n    if decision is None:\n        raise ValueError(\'Offline replay needs completed stage-one execution\')\n    if not decision:\n        record.update(main=main[:k2], long=long[:k2])\n    winner, count = select_available(record, k2)\n    return winner, decision, count\n', 'final_speed.py': '"""Opt-in decoder experiments. Frozen final_fast/final_decode remain the control."""\nfrom dataclasses import dataclass\nimport torch\nfrom torch.nn import functional as F\nimport final_core as C\nimport final_fast as Fast\n\n\nGRAPH_REFUSAL = (\n    \'CUDA graphs disabled: prefix length, KV slice addresses and active batch change every step. \'\n    \'One graph per position/batch would require hundreds of captures; full-capacity masked SDPA \'\n    \'changes reduction shapes and cannot promise bit-exact greedy. No untested graph fallback.\')\n\n\n@dataclass(frozen=True)\nclass Options:\n    preallocated_kv: bool = True  # Already present in the frozen baseline; no new speedup.\n    share_cross: bool = False\n    async_eos: bool = False\n    eos_every: int = 16\n    compact_done: bool = False\n    compact_fraction: float = .25\n    race_sampler: bool = False\n    lean_greedy: bool = False\n    cuda_graphs: bool = False\n\n    def __post_init__(self):\n        if not self.preallocated_kv:\n            raise ValueError(\'Baseline already preallocates KV; a cat regression is not an optimization\')\n        if self.cuda_graphs:\n            raise NotImplementedError(GRAPH_REFUSAL)\n        if self.eos_every < 1 or not 0 < self.compact_fraction <= 1:\n            raise ValueError(\'Invalid EOS/compaction interval\')\n\n\nclass Decoder(Fast.Decoder):\n    @torch.inference_mode()\n    def __init__(self, context, k, share_cross=False):\n        if not share_cross or k != 1:\n            super().__init__(context, k)\n            return\n        # Immutable cross KV and source mask can be shared for greedy, no replication.\n        self.c, self.k, self.t, self.n = context, k, 0, len(context.src)\n        self.allowed, self.cross = context.allowed, context.cross\n        self.cross_bytes = sum(t.numel()*t.element_size() for pair in self.cross for t in pair)\n        shape = (self.n, context.heads, context.capacity, context.dim)\n        self.self_kv = [(context.positions.new_empty(shape), context.positions.new_empty(shape))\n                        for _ in context.cross]\n        self.valid = torch.zeros(self.n, context.capacity, dtype=torch.bool, device=context.src.device)\n\n    @torch.inference_mode()\n    def select(self, index):\n        """Copy only initialized self-KV prefix, then continue at the SAME position."""\n        if index.ndim != 1 or not len(index):\n            raise ValueError(\'Nonempty 1-D active indices required\')\n        n, cap = len(index), self.c.capacity\n        self.allowed = self.allowed.index_select(0, index)\n        self.cross = [tuple(t.index_select(0, index) for t in pair) for pair in self.cross]\n        # Release each old layer as soon as its replacement is ready. Holding both\n        # complete caches would make compaction itself an avoidable T4 OOM peak.\n        for number in range(len(self.self_kv)):\n            pair = self.self_kv[number]\n            new_pair = []\n            for old in pair:\n                new = old.new_empty((n, self.c.heads, cap, self.c.dim))\n                new[:, :, :self.t].copy_(old[:, :, :self.t].index_select(0, index))\n                new_pair.append(new)\n            self.self_kv[number] = tuple(new_pair)\n        self.n = n\n        self.valid = self.valid.index_select(0, index)\n\n\ndef race_pick(probabilities, noise):\n    """Independent Exp(1) race: argmax(p/E) has categorical probabilities p.\n\n    Uses exactly the same FP32 softmax as baseline; RNG consumption can change.\n    Avoids multinomial\'s probability validation/reductions. Noise storage is reused.\n    """\n    noise.exponential_()\n    return torch.div(probabilities, noise, out=noise).argmax(-1)\n\n\n@torch.inference_mode()\ndef decode(context, k=1, temperature=None, deadline=None, compute_logprob=True,\n           options=None, stats=None):\n    options = options or Options()\n    d = Decoder(context, k, options.share_cross)\n    device, n = context.src.device, d.n\n    token = torch.full((n,), C.BOS, device=device, dtype=torch.long)\n    done = torch.zeros(n, dtype=torch.bool, device=device)\n    output = torch.full((n, context.capacity), C.PAD, device=device, dtype=torch.long)\n    sums, lengths = torch.zeros(n, device=device), torch.zeros(n, dtype=torch.long, device=device)\n    final_sums, final_lengths = torch.zeros_like(sums), torch.zeros_like(lengths)\n    original = torch.arange(n, device=device)\n    compacted = False\n    noise = None\n    host = event = None\n    pending = False\n    async_eos = options.async_eos and device.type == \'cuda\'\n    if async_eos:\n        host = torch.empty(n, dtype=torch.bool, pin_memory=True)\n        event = torch.cuda.Event()\n    history = []\n    checks = 0\n\n    def flush():\n        final_sums.index_copy_(0, original, sums)\n        final_lengths.index_copy_(0, original, lengths)\n\n    def inspect(snapshot):\n        nonlocal token, done, sums, lengths, original, compacted, noise, host\n        if bool(snapshot.all()):\n            return True\n        if not options.compact_done:\n            return False\n        keep = (~snapshot).nonzero().flatten()  # CPU only: never CUDA nonzero/item.\n        if options.compact_done and len(keep) <= d.n*(1-options.compact_fraction):\n            flush()\n            index = keep.to(device)\n            before = d.n\n            d.select(index)\n            token, done = token[index], done[index]\n            sums, lengths, original = sums[index], lengths[index], original[index]\n            compacted, noise = True, None\n            history.append({\'step\': d.t, \'before\': before, \'after\': d.n})\n            if async_eos:\n                host = torch.empty(d.n, dtype=torch.bool, pin_memory=True)\n        return False\n\n    try:\n        for t in range(context.capacity):\n            if t % 16 == 0:\n                C.check_deadline(deadline)\n            if pending and event.query():\n                pending = False\n                if inspect(host):\n                    break\n            logits = d.step(token)\n            if temperature is not None or compute_logprob:\n                logits = logits.float()\n            logits[:, :2] = -torch.inf\n            if temperature is None:\n                pick = logits.argmax(-1)\n            else:\n                probabilities = F.softmax(logits / temperature, -1)\n                if options.race_sampler:\n                    if noise is None:\n                        noise = torch.empty_like(probabilities)\n                    pick = race_pick(probabilities, noise)\n                else:\n                    pick = torch.multinomial(probabilities, 1).squeeze(-1)\n            if compute_logprob:\n                lp = F.log_softmax(logits, -1).gather(-1, pick[:, None]).squeeze(-1)\n                sums.add_(torch.where(done, 0., lp))\n            if compute_logprob or not options.lean_greedy:\n                lengths.add_(~done)\n            token = pick.masked_fill(done, C.PAD)\n            if compacted:\n                output[:, t].index_copy_(0, original, token)\n            else:\n                output[:, t] = token\n            done |= token.eq(C.EOS)\n            if (t+1) % options.eos_every == 0:\n                if async_eos:\n                    if not pending:\n                        host.copy_(done, non_blocking=True)\n                        event.record()\n                        pending = True\n                        checks += 1\n                else:\n                    checks += 1\n                    if inspect(done.cpu()):\n                        break\n    finally:\n        # Also retain pinned storage on deadline/OOM exceptions until copy finishes.\n        if pending:\n            event.synchronize()\n    flush()\n    if stats is not None:\n        stats.update(steps=d.t, eos_checks=checks, compactions=history,\n                     async_eos=async_eos, rng_stream_may_change=options.race_sampler or options.compact_done)\n    return (output[:, :d.t].reshape(len(context.src), k, d.t).cpu(),\n            (final_sums / final_lengths.clamp_min(1)).reshape(-1, k).cpu(), d.cross_bytes)\n', 'final_execute_v6.py': '"""Torch-free v6 service. Same isolated worker, nice10, live 2 -> up to4 slots."""\nimport json\nimport multiprocessing as mp\nimport os\nimport queue\nimport signal\nimport sys\nimport threading\nimport time\nfrom collections import deque\nfrom multiprocessing.connection import wait\nfrom final_execute import worker, WALL\n\n\ndef lower_priority():\n    error = None\n    try:\n        effective = os.nice(10)\n    except PermissionError as exc:\n        effective, error = os.getpriority(os.PRIO_PROCESS, 0), str(exc)\n    return {\'nice_requested_increment\': 10, \'nice\': effective,\n            \'nice_applied\': error is None, \'nice_error\': error}\n\n\ndef serve(workers, priority):\n    inbox = queue.Queue()\n    def read():\n        for line in sys.stdin:\n            inbox.put(json.loads(line))\n        inbox.put(None)\n    threading.Thread(target=read, daemon=True).start()\n    context = mp.get_context(\'spawn\')\n    slots = []\n    pending = deque()\n    finished = False\n    def start():\n        parent, child = context.Pipe()\n        process = context.Process(target=worker, args=(child,), daemon=True)\n        process.start()\n        child.close()\n        return [process, parent, None, 0.]\n    def stop(slot):\n        process, pipe, _, _ = slot\n        if process.is_alive():\n            process.kill()\n        process.join(timeout=.05)\n        pipe.close()\n    def emit(index, result, metrics=None):\n        print(json.dumps([index, result, metrics]), flush=True)\n    try:\n        slots = [start() for _ in range(workers)]\n        def state():\n            print(json.dumps({\'workers\': len(slots), **priority}), flush=True)\n        state()\n        while not finished or pending or any(s[2] is not None for s in slots):\n            while True:\n                try:\n                    batch = inbox.get_nowait()\n                except queue.Empty:\n                    break\n                if batch is None:\n                    finished = True\n                elif isinstance(batch, dict):\n                    target = batch[\'workers\']\n                    if not isinstance(target, int) or not len(slots) <= target <= 4:\n                        raise ValueError(\'Only expansion up to four workers is supported\')\n                    slots.extend(start() for _ in range(target-len(slots)))\n                    state()\n                else:\n                    pending.extend(batch)\n            for number, slot in enumerate(slots):\n                process, pipe, active, submitted = slot\n                pipe_failed = False\n                # Worker enforces wall+2; allow a bounded IPC/reap margin here.\n                try:\n                    if active is not None and pipe.poll():\n                        index, result, *metadata = pipe.recv()\n                        assert index == active\n                        emit(index, result, metadata[0] if metadata else None)\n                        slot[2] = None\n                except (EOFError, OSError):\n                    pipe_failed = True\n                if pipe_failed or not process.is_alive() or (slot[2] is not None and time.monotonic()-submitted >= WALL+2.5):\n                    if slot[2] is not None:\n                        emit(slot[2], {\'status\': \'timeout_or_worker_exit\'})\n                    stop(slot)\n                    slots[number] = slot = start()\n                    process, pipe = slot[:2]\n                if slot[2] is None and pending:\n                    index, code = pending.popleft()\n                    try:\n                        pipe.send((index, code))\n                        slot[2:] = [index, time.monotonic()]\n                    except (EOFError, OSError):\n                        # Death can race is_alive(): this job was not accepted.\n                        pending.appendleft((index, code))\n                        stop(slot)\n                        slots[number] = start()\n            active_pipes = [slot[1] for slot in slots if slot[2] is not None]\n            if active_pipes:\n                wait(active_pipes, timeout=.002)  # Wake immediately, no per-program 1ms polling tax.\n            else:\n                time.sleep(.002)\n    finally:\n        for slot in slots:\n            stop(slot)\n\n\nif __name__ == \'__main__\':\n    if sys.platform.startswith(\'linux\'):\n        import ctypes\n        parent = os.getppid()\n        signal.signal(signal.SIGTERM, lambda *_: os.killpg(os.getpgrp(), signal.SIGKILL))\n        assert ctypes.CDLL(None).prctl(1, signal.SIGTERM, 0, 0, 0) == 0\n        if parent == 1 or os.getppid() != parent:\n            os.killpg(os.getpgrp(), signal.SIGKILL)\n    serve(int(sys.argv[1]), lower_priority())\n', 'final_executor_v6.py': '"""V6 execution cache with nice10 service and lossless 2 -> up to4 expansion."""\nimport math\nimport json\nimport os\nimport signal\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\n\n\nclass Executor:\n    def __init__(self, workers=None, service_path=None):\n        self.workers = workers if workers is not None else 2\n        if isinstance(self.workers, bool) or not isinstance(self.workers, int) or not 1 <= self.workers <= 4:\n            raise ValueError(\'Require one to four execution workers\')\n        env = dict(os.environ, OMP_NUM_THREADS=\'1\', MKL_NUM_THREADS=\'1\', OPENBLAS_NUM_THREADS=\'1\',\n                   PYTHONHASHSEED=\'0\')\n        self.child = subprocess.Popen([sys.executable, str(service_path or Path(__file__).with_name(\'final_execute_v6.py\')),\n                                      str(self.workers)], stdin=subprocess.PIPE, stdout=subprocess.PIPE,\n                                      stderr=subprocess.DEVNULL, text=True, bufsize=1, env=env,\n                                      start_new_session=True)\n        self.service_state = {}\n        self.worker_history = [self.workers]\n        self.codes = {}\n        self.results = {}\n        self.requests = 0\n        self.started = time.monotonic()\n        self.error = None\n        self.task_wall_seconds = self.task_cpu_seconds = 0.\n        self.first_submit = self.last_result = None\n        def read():\n            try:\n                for line in self.child.stdout:\n                    message = json.loads(line)\n                    if isinstance(message, dict):\n                        self.service_state = message\n                        continue\n                    index, value, *metadata = message\n                    if metadata and metadata[0]:\n                        self.task_wall_seconds += metadata[0][\'task_wall_seconds\']\n                        self.task_cpu_seconds += metadata[0][\'task_cpu_seconds\']\n                    self.last_result = time.monotonic()\n                    self.results[index] = value\n            except Exception as error:\n                self.error = type(error).__name__\n        self.reader = threading.Thread(target=read, daemon=True)\n        self.reader.start()\n\n    def resize(self, workers):\n        if not isinstance(workers, int) or not self.workers <= workers <= 4:\n            raise ValueError(\'Only expansion up to four workers is supported\')\n        self.child.stdin.write(json.dumps({\'workers\': workers})+\'\\n\')\n        self.child.stdin.flush()\n        self.workers = workers\n        self.worker_history.append(workers)\n\n    def submit(self, candidates):\n        jobs = []\n        for candidate in candidates:\n            self.requests += 1\n            code = candidate[\'code\']\n            if code not in self.codes:\n                index = len(self.codes)\n                self.codes[code] = index\n                jobs.append([index, code])\n        if jobs:\n            if self.first_submit is None:\n                self.first_submit = time.monotonic()\n            self.child.stdin.write(json.dumps(jobs) + \'\\n\')\n            self.child.stdin.flush()\n\n    def attach(self, candidates):\n        for candidate in candidates:\n            index = self.codes[candidate[\'code\']]\n            candidate[\'execution\'] = self.results.get(index, {\'status\': \'budget_not_evaluated\'})\n\n    def drain(self, deadline):\n        while (len(self.results) < len(self.codes) or self.service_state.get(\'workers\') != self.workers) and time.monotonic() < deadline:\n            exit_code = self.child.poll()\n            if exit_code is not None:\n                self.error = self.error or \'executor_exit_\'+str(exit_code)\n                break\n            if self.error:\n                break\n            time.sleep(.002)\n        return len(self.results) == len(self.codes) and self.service_state.get(\'workers\') == self.workers\n\n    def summary(self):\n        return {\'workers\': self.workers, \'worker_history\': self.worker_history,\n                \'service_state\': dict(self.service_state), \'requested_candidates\': self.requests,\n                \'unique_programs\': len(self.codes), \'deduplicated\': self.requests-len(self.codes),\n                \'completed_unique\': len(self.results), \'service_wall_seconds\': time.monotonic()-self.started,\n                \'sum_worker_task_wall_seconds\': self.task_wall_seconds,\n                \'sum_worker_task_cpu_seconds\': self.task_cpu_seconds,\n                \'first_submit_to_last_result_seconds\': (self.last_result-self.first_submit\n                    if self.last_result is not None and self.first_submit is not None else None),\n                \'service_wall_includes_GPU_wait\': True,\n                \'wall_limit\': 3., \'cpu_limit\': 2., \'memory_bytes\': 768*1024**2,\n                \'overlap\': True, \'isolation\': \'fresh numeric namespaces / pristine SymPy fork\', \'error\': self.error}\n\n    def close(self):\n        # Never pool.join() unbounded in the CUDA process.\n        try:\n            os.killpg(self.child.pid, signal.SIGKILL)\n        except ProcessLookupError:\n            pass\n        try:\n            self.child.wait(timeout=.1)\n        except subprocess.TimeoutExpired:\n            pass\n        self.child.stdin.close()\n        self.reader.join(timeout=.1)\n        self.child.stdout.close()\n\n\ndef verify_winners(winners, deadline, workers=None):\n    """Fresh service: compare the actual chosen CODE against its voted execution."""\n    executor = Executor(workers)\n    eligible = [i for i, w in enumerate(winners) if w[\'execution\'][\'status\'] == \'numeric\']\n    items = [{\'code\': winners[i][\'code\']} for i in eligible]\n    try:\n        executor.submit(items)\n        complete = executor.drain(deadline)\n        executor.attach(items)\n        mismatches = [i for i, item in zip(eligible, items) if\n                      item[\'execution\'][\'status\'] != \'numeric\' or not math.isclose(\n                          item[\'execution\'][\'value\'], winners[i][\'execution\'][\'value\'],\n                          rel_tol=1e-6, abs_tol=1e-6)]\n        return {\'checked_rows\': len(eligible), \'complete\': complete,\n                \'mismatch_rows\': mismatches, \'execution\': executor.summary()}\n    finally:\n        executor.close()\n', 'qk_attention.py': '"""Headwise Q/K LayerNorm before scaled SDPA; legacy MHA parameter names retained."""\nimport math\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef norm_q(a, x):\n    return a.q_norm(x) if hasattr(a, \'q_norm\') else x\n\n\ndef norm_k(a, x):\n    return a.k_norm(x) if hasattr(a, \'k_norm\') else x\n\n\nclass HeadLayerNorm(nn.LayerNorm):\n    def forward(self, x):\n        # FP32 statistics/affines also when inference casts the entire model to FP16.\n        return F.layer_norm(x.float(),self.normalized_shape,self.weight.float(),self.bias.float(),self.eps).to(x.dtype)\n\n\nclass QKAttention(nn.MultiheadAttention):\n    def forward(self, query, key, value, key_padding_mask=None, need_weights=True,\n                attn_mask=None, average_attn_weights=True, is_causal=False):\n        assert self.batch_first and query.ndim == 3 and self._qkv_same_embed_dim\n        assert not self.add_zero_attn and self.bias_k is None and self.bias_v is None\n        width=self.embed_dim; n,t,_=query.shape; length=key.shape[1]; dim=width//self.num_heads\n        wq,wk,wv=self.in_proj_weight.chunk(3)\n        bq,bk,bv=self.in_proj_bias.chunk(3) if self.in_proj_bias is not None else (None,)*3\n        def heads(x):return x.reshape(n,-1,self.num_heads,dim).transpose(1,2)\n        q=norm_q(self,heads(F.linear(query,wq,bq)));k=norm_k(self,heads(F.linear(key,wk,bk)))\n        v=heads(F.linear(value,wv,bv));mask=None\n        if attn_mask is not None:\n            mask=attn_mask\n            if mask.dtype==torch.bool:mask=torch.zeros_like(mask,dtype=q.dtype).masked_fill(mask,float(\'-inf\'))\n            if mask.ndim==2:mask=mask[None,None]\n            elif mask.ndim==3:mask=mask.reshape(n,self.num_heads,t,length)\n        elif is_causal:\n            mask=torch.full((t,length),float(\'-inf\'),dtype=q.dtype,device=q.device).triu(1)[None,None]\n        if key_padding_mask is not None:\n            padding=key_padding_mask\n            if padding.dtype==torch.bool:padding=torch.zeros_like(padding,dtype=q.dtype).masked_fill(padding,float(\'-inf\'))\n            padding=padding[:,None,None,:];mask=padding if mask is None else mask+padding\n        if getattr(self,\'record_logits\',False):\n            # Cheap sampled maximum: first batch, all heads, first eight queries, all keys.\n            with torch.no_grad(),torch.autocast(device_type=q.device.type,enabled=False):\n                sample=(q[:1,:,:8].float()@k[:1].float().transpose(-1,-2))/math.sqrt(dim)\n                self.sampled_logit_max=sample.abs().amax().detach()\n        p=self.dropout if self.training else 0.\n        if need_weights:\n            weights=(q@k.transpose(-1,-2))/math.sqrt(dim)\n            if mask is not None:weights=weights+mask\n            weights=F.dropout(weights.softmax(-1),p,training=self.training)\n            y=weights@v;returned=weights.mean(1) if average_attn_weights else weights\n        else:\n            y=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,dropout_p=p);returned=None\n        return self.out_proj(y.transpose(1,2).contiguous().reshape(n,t,width)),returned\n\n\ndef install_qk_norm(transformer, flag):\n    if flag in (False,None,\'none\'):return\n    if flag not in (True,\'layernorm\'):raise ValueError(\'qk_norm must be false or layernorm\')\n    for layer in [*transformer.encoder.layers,*transformer.decoder.layers]:\n        # Disable PyTorch\'s fused encoder path, which would bypass the QKAttention forward.\n        if isinstance(layer,nn.TransformerEncoderLayer):layer.activation_relu_or_gelu=0\n        for a in (layer.self_attn, *([layer.multihead_attn] if hasattr(layer,\'multihead_attn\') else [])):\n            a.__class__=QKAttention\n            a.q_norm=HeadLayerNorm(a.head_dim,eps=1e-5)\n            a.k_norm=HeadLayerNorm(a.head_dim,eps=1e-5)\n', 'final_ensemble.py': '"""Ordered 2–4 model ensemble; K2 is the TOTAL sample limit for each model.\n\nDecisions use executed programs only. Flattening preserves model/lane order and\ndelegates voting/code selection to the frozen v3–v6 implementation.\n"""\nimport math\nfrom pathlib import Path\nimport re\nfrom final_adaptive import numeric, same, select\n\nDEFAULT_MODELS = [\n    {\'name\': \'C-main\', \'filename\': \'model-C-main__weights__l2-62280-fp16.pt\',\n     \'sha256\': \'8c11b1038855422b6912e243049a554317978f848edb412ed48aebbe4cb6b886\',\n     \'dataset\': \'thisarivson/nlp-hw02-l3-live\'},\n    {\'name\': \'C-long2x\', \'filename\': \'model-C-long2x__weights__l2-62280-fp16.pt\',\n     \'sha256\': \'d10bf0018d0b14fd6cc3c8b9c7121f127b5cec97057544fd63e5824253a55057\',\n     \'dataset\': \'thisarivson/nlp-hw02-l3-live\'},\n    {\'name\': \'pilot-b\', \'filename\': \'model-pilot-b__weights__long-truncation-fp16.pt\',\n     \'sha256\': \'8439037720dda16dda6b73fe676860d5498bbcba4cacebc0d709bbb22db464a5\',\n     \'dataset\': \'thisarivson/nlp-hw02-p1-weights\'},\n]\n\n\ndef parameters(config):\n    models = config.get(\'models\')\n    if not isinstance(models, list) or not 2 <= len(models) <= 4:\n        raise ValueError(\'Require an ordered list of 2–4 models\')\n    normalized = []\n    for model in models:\n        if not isinstance(model, dict):\n            raise ValueError(\'Require model objects with name/filename/sha256/K1/K2\')\n        name, filename, digest = (model.get(k) for k in (\'name\', \'filename\', \'sha256\'))\n        if not isinstance(name, str) or not re.fullmatch(r\'[A-Za-z0-9_-]+\', name) or name == \'greedy\':\n            raise ValueError(\'Invalid model name\')\n        if (not isinstance(filename, str) or Path(filename).name != filename or\n                not filename.endswith(\'.pt\') or \'\\\\\' in filename):\n            raise ValueError(\'Require a weight basename ending in .pt\')\n        if not isinstance(digest, str) or not re.fullmatch(r\'[a-f0-9]{64}\', digest):\n            raise ValueError(\'Every model requires a pinned SHA256\')\n        k1, k2 = model.get(\'K1\', config.get(\'K1\', 1)), model.get(\'K2\', config.get(\'K2\', 4))\n        if (isinstance(k1, bool) or not isinstance(k1, int) or k1 < 1 or\n                isinstance(k2, bool) or not isinstance(k2, int) or k2 < k1):\n            raise ValueError(\'Require integer 1 <= K1 <= K2 for each model\')\n        if \'dataset\' in model and (not isinstance(model[\'dataset\'], str) or\n                not re.fullmatch(r\'[A-Za-z0-9_-]+/[A-Za-z0-9_-]+\', model[\'dataset\'])):\n            raise ValueError(\'Invalid model dataset\')\n        normalized.append(dict(model, K1=k1, K2=k2))\n    if len({m[\'name\'] for m in normalized}) != len(models):\n        raise ValueError(\'Model names must be unique\')\n    threshold = config.get(\'unanimity_threshold\', 1.)\n    if (isinstance(threshold, bool) or not isinstance(threshold, (int, float)) or\n            not math.isfinite(threshold) or not 0 < threshold <= 1):\n        raise ValueError(\'Require 0 < unanimity_threshold <= 1\')\n    return normalized, float(threshold)\n\n\ndef candidates(record, models, stage=\'K2\'):\n    return [record[\'greedy\']]+[c for m in models for c in record[\'samples\'][m[\'name\']][:m[stage]]]\n\n\ndef stage1_decision(record, models, threshold=1.):\n    if record[\'greedy\'] is None or any(len(record[\'samples\'][m[\'name\']]) < m[\'K1\'] for m in models):\n        return None\n    first = candidates(record, models, \'K1\')\n    if any(c.get(\'execution\', {}).get(\'status\', \'budget_not_evaluated\') == \'budget_not_evaluated\' for c in first):\n        return None\n    if not all(numeric(c) for c in first):\n        return False\n    return sum(same(c[\'execution\'][\'value\'], first[0][\'execution\'][\'value\']) for c in first)/len(first) >= threshold\n\n\ndef stage1_leader_fraction(record, models):\n    if stage1_decision(record, models) is None:\n        raise ValueError(\'Priority requires completed stage-one executions\')\n    first = candidates(record, models, \'K1\')\n    remaining = [c for c in first if numeric(c)]\n    largest = 0\n    while remaining:\n        component = [remaining.pop(0)]\n        while True:\n            adjacent = [c for c in remaining if any(same(c[\'execution\'][\'value\'], v[\'execution\'][\'value\']) for v in component)]\n            if not adjacent:\n                break\n            component.extend(adjacent)\n            identities = {id(c) for c in adjacent}\n            remaining = [c for c in remaining if id(c) not in identities]\n        largest = max(largest, len(component))\n    return largest/len(first)\n\n\ndef select_available(record, models, **kwargs):\n    available = [c for c in candidates(record, models)[1:] if\n                 c.get(\'execution\', {}).get(\'status\', \'budget_not_evaluated\') != \'budget_not_evaluated\']\n    return select(record[\'greedy\'], available, [], k=len(available), **kwargs), len(available)\n\n\ndef replay_record(greedy, samples, models, threshold=1.):\n    record = {\'greedy\': greedy, \'samples\': {m[\'name\']: samples[m[\'name\']][:m[\'K1\']] for m in models}}\n    decision = stage1_decision(record, models, threshold)\n    if decision is None:\n        raise ValueError(\'Offline replay requires completed stage-one executions\')\n    if not decision:\n        record[\'samples\'] = {m[\'name\']: samples[m[\'name\']][:m[\'K2\']] for m in models}\n    winner, count = select_available(record, models)\n    return winner, decision, count\n', 'final_mounts.py': '"""Resolve dataset and kernel-output mounts without hardcoded owner layouts."""\nfrom pathlib import Path\n\n\ndef find_input(slug, root=Path(\'/kaggle/input\')):\n    root = Path(root)\n    if (root/slug).is_dir():\n        return root/slug\n    hits = [p for p in root.rglob(slug) if p.is_dir()] if root.is_dir() else []\n    assert len(hits) == 1, f\'Cannot resolve input {slug}: {[str(p) for p in hits]}\'\n    return hits[0]\n\n\ndef resolve_cache_mount(input_root, source, kind):\n    # Same precedence as l3_resume.resolve_cache_mount; no training imports.\n    root = Path(input_root)\n    owner, slug = source.split(\'/\')[:2]\n    assert kind in (\'datasets\', \'notebooks\')\n    for candidate in (root/kind/owner/slug, root/slug):\n        if candidate.is_dir():\n            return candidate\n    return find_input(slug, root)\n'}
for name,source in SOURCES.items():(MODULES/name).write_text(source)
sys.path.insert(0,str(MODULES))
from final_session import Watchdog
WATCHDOG=Watchdog(OUTPUT,SESSION_START).arm()


In [ ]:
BENCHMARK=True  # Ровно один холодный test1919 + tune2048.
from final_mounts import find_input, resolve_cache_mount
INPUTS_ROOT=find_input('nlp-hw02-26b-inputs')
ENSEMBLE_CONFIG={'models': [{'name': 'C-long2x-B6', 'filename': 'b6-final-fp16.pt', 'relative_path': 'model-GPU1/weights/b6-final-fp16.pt', 'sha256': 'ee30794048bb6200cf97ac4d7e5c1606f0e97893ea04f6567c9487f6c04345fa', 'dataset': 'friend1/nlp-hw02-b6', 'source_kind': 'notebooks', 'source_ref': 'friend1/nlp-hw02-b6/1', 'K1': 1, 'K2': 1}, {'name': 'pilot-b', 'filename': 'long-truncation-fp16.pt', 'sha256': '8439037720dda16dda6b73fe676860d5498bbcba4cacebc0d709bbb22db464a5', 'dataset': 'friend1/nlp-hw02-kaggle-p1', 'relative_path': 'model-pilot-b/weights/long-truncation-fp16.pt', 'source_kind': 'notebooks', 'source_ref': 'friend1/nlp-hw02-kaggle-p1/3', 'K1': 1, 'K2': 8}, {'name': 'W-B6', 'filename': 'b6-final-fp16.pt', 'relative_path': 'model-GPU0/weights/b6-final-fp16.pt', 'sha256': 'cb34c81e6cda83c73bf4662c5c425446be8cdb48c30dd5e9e217fd6cbc1cd496', 'dataset': 'friend1/nlp-hw02-b6', 'source_kind': 'notebooks', 'source_ref': 'friend1/nlp-hw02-b6/1', 'K1': 1, 'K2': 8}], 'unanimity_threshold': 1.0, 'sequence_budget': 1024, 'greedy_sequence_budget': 1024, 'scorer_enabled': False, 'scorer_weight': 0.0, 'expected_matches': 1553}
WEIGHTS_ROOTS={m['dataset']:resolve_cache_mount(Path('/kaggle/input'),m['dataset'],m.get('source_kind','datasets')) for m in ENSEMBLE_CONFIG['models']}
from final_session import suite
REPORT=suite(INPUTS_ROOT,WEIGHTS_ROOTS,OUTPUT,MODULES,WATCHDOG,benchmark=BENCHMARK,ensemble_config=ENSEMBLE_CONFIG)
print(json.dumps({'status':REPORT['status'],'test_seconds':[r['full_pipeline_seconds'] for r in REPORT['test']],
                 'quality':REPORT.get('quality'),'gates':REPORT.get('gates'),'session_seconds':REPORT['session_seconds']},indent=2))


Скачать четыре файла, указанные выше. Проверить status, gates, test time, guard/fallback/partial, реальные SHA весов и порядок строк. Review_required из-за quality — диагностический результат; политика после запуска не меняется.